<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/AiZynthFinder_with_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**In the top Colab menu, click Runtime > Change runtime type and select T4 GPU** (Medusa runs on PyTorch GPU; the paper reports ~0.16–0.31 seconds per Retro* iteration).

## Exploring speculative beam search

[Fast and scalable retrosynthetic planning with a transformer neural network and speculative beam search](https://doi.org/10.1039/d5dd00573f) ([arXiv:2508.01459](https://arxiv.org/abs/2508.01459)) explores faster computer-aided synthesis planning in AiZynthFinder.

### Core concepts

- **Single-step model:** A SMILES-to-SMILES transformer predicts reactant sets for a target molecule.
- **Medusa speculative beam search:** Lightweight drafting heads propose future tokens for multiple beams; the transformer verifies those proposals in parallel instead of generating every token sequentially. This reduces model forward passes and decoding wall time while retaining beam-search scoring for verified candidates.
- **Planning integration:** AiZynthFinder consumes Medusa expansions through its transformer policy and searches for routes with Retro*.

### Reported results and use cases

The paper reports approximately **91% speculative-step acceptance** and **26%–86% more molecules solved under the same 5–15-second time budgets** in its benchmark settings, including USPTO50K and Caspyrus10k. Faster expansions could help high-throughput synthesizability screening and de novo design; these numbers are benchmark results, not guaranteed gains for this notebook or its stock and targets.

### Implementation Approach

This notebook uses the authors' Medusa model and Retro* planner:

1. **Prepare a GPU runtime and models (Cell 1).** Install the [paper's implementation repository](https://github.com/Academich/faster-ml-casp) and download its Medusa checkpoint, vocabulary, and building-block stock. The authors' downloader saves ZINC at `data/bb_stock/zinc_stock.hdf5`.
2. **Initialize the planner (Cell 3).** Load the repository's [full Medusa config](https://github.com/Academich/faster-ml-casp/blob/main/configs/medusa_default_config.yml) with `sampling: medusa`, select `finder.expansion_policy.select("transformer")`, and use Retro* with ZINC and optional custom-lab stock. No template filter is selected.
3. **Tune the wall-clock budget (Cells 5 and 5B).** Select a preset or adjust `time_limit`, `iteration_limit`, `return_first`, and `beam_size` before running the search button. Whichever limit is hit first stops the search; a beam of 10 prioritizes speed, while 50 explores more candidate reactants.

**Expected effect:** On a suitable GPU, faster single-step decoding can let the tree search examine more expansions before its time limit and potentially find more complete routes. It does not by itself improve chemical validity or guarantee better-ranked routes; replacing the notebook's USPTO template policy with a transformer may also change the candidate reactions, so differences from the original notebook cannot be attributed solely to speculative decoding. When a low iteration limit binds first, the wall-time advantage is unlikely to increase the number solved.


In [ ]:
# @title 1. Install Dependencies & Download Medusa + Stock Models
import os
import sys

if not os.path.isdir("faster-ml-casp"):
    !git clone -q https://github.com/Academich/faster-ml-casp.git

repo_path = os.path.abspath("faster-ml-casp")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

!pip install -q rdkit onnxruntime tables scipy pydantic deprecated jinja2 networkx pillow requests tqdm pyyaml rdchiral paretoset marshmallow jedi Reaction-Utils google-genai markdown gdown pandas psutil registry-factory==0.1.2 'lightning>=2.6.6'
!pip install -q -e faster-ml-casp --no-deps

stock_files = ("data/bb_stock/zinc_stock.hdf5", "data/bb_stock/paroutes_n1_stock.hdf5")
if not all(os.path.isfile(path) for path in stock_files):
    !mkdir -p data && python3 faster-ml-casp/aizynthfinder/tools/download_public_data.py data
if not all(os.path.isfile(path) for path in stock_files):
    raise FileNotFoundError("Stock download incomplete; rerun Cell 1 before initializing Medusa.")

medusa_files = (
    "checkpoints/retro/medusa/6_epochs_lr3e_4_then_1ep_lr3e_5.ckpt",
    "checkpoints/retro/medusa/vocab_trained_on_USPTO_50K_PtoR_aug20.json",
)
if not all(os.path.isfile(path) for path in medusa_files):
    !mkdir -p checkpoints/retro/medusa
    !gdown --folder https://drive.google.com/drive/folders/1uE8J13AgPpfLRJuGuXBOJS1aP2IA1fTk -O checkpoints/retro/medusa
if not all(os.path.isfile(path) for path in medusa_files):
    raise FileNotFoundError("Medusa checkpoint or vocabulary missing; rerun Cell 1 before initializing Medusa.")
else:
    print("✅ Medusa checkpoint and vocabulary ready.")

In [ ]:
# @title 2. Core Definitions: SMARTS Library, Reaction Classifiers & Gemini Helpers
import os
import json
import logging
from pydantic import BaseModel
from rdkit import Chem
from google import genai
from google.genai import types

# --- 1. API Key & Client Singleton ---
def get_gemini_client():
    """Returns a cached genai.Client instance if KDD_GEMINI_API_KEY is available."""
    if "_GEMINI_CLIENT" in globals() and globals()["_GEMINI_CLIENT"] is not None:
        return globals()["_GEMINI_CLIENT"]

    api_key = None
    try:
        from google.colab import userdata
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if api_key:
        globals()["_GEMINI_CLIENT"] = genai.Client(api_key=api_key)
        return globals()["_GEMINI_CLIENT"]
    return None

# --- 2. Medicinal Chemistry SMARTS Dictionary ---
# Pipe '|' separates sub-patterns so RDKit compiles each valid SMARTS string into a list
RAW_SMARTS = {
    "boronic": "[#5](O)O|[#5]1O[#6][#6]O1",
    "stannane": "[#50]",
    "zinc_magnesium": "[#12,#30]",
    "silicon_coupling": "[#14](F)|[#14](O)",
    "terminal_alkyne": "[CX2]#[CX2H1]",
    "internal_alkyne": "[CX2]#[CX2;!H1]",
    "alkene": "[CX3]=[CX3]",
    "aryl_halide_pseudohalide": "c[Cl,Br,I]|cOS(=O)(=O)C(F)(F)F",
    "alkyl_lg": "[CX4][Cl,Br,I]|[#6]OS(=O)(=O)[#6]",
    "carboxylic_acid": "[CX3](=O)[OX2H1]",
    "ester": "[#6][CX3](=O)[OX2][#6]",
    "benzyl_ester": "[#6][CX3](=O)[OX2]Cc1ccccc1",
    "acyl_halide": "[CX3](=O)[Cl,Br,F]",
    "anhydride": "[CX3](=O)O[CX3](=O)",
    "amide": "[#6][CX3](=O)[NX3]",
    "primary_amide": "[CX3](=O)[NX3H2]",
    "thioamide": "[#6](=[#16])[#7]",
    "amine_1_2": "[NX3;H2,H1;!$(NC=O);!$(NS=O)]",
    "tert_amine": "[NX3;H0;!$(NC=O);!$(NS=O)]",
    "aryl_amine": "c[NX3]",
    "nitro": "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
    "nitrile": "[CX2]#[NX1]",
    "azide": "[NX2]=[NX2+]=[NX1-]|[NX1-]=[NX2+]=[NX2]",
    "triazole": "c1ncnn1|c1nnn[nH]1",
    "tetrazole": "c1nnnn1",
    "thiazole": "c1ncsc1",
    "imidazole_oxazole": "c1nc[nH,o]c1",
    "indole": "c1ccc2[nH]ccc2c1",
    "aldehyde_ketone": "[CX3H1](=O)[#6]|[CX3](=O)([#6])[#6]",
    "alcohol": "[CX4][OX2H1]",
    "phenol": "c[OX2H1]",
    "ether": "[#6][OX2][#6]",
    "aryl_ether": "c[OX2][CX4]",
    "epoxide": "C1OC1",
    "sulfonamide": "S(=O)(=O)[NX3]",
    "sulfonyl_chloride": "S(=O)(=O)[Cl]",
    "thiol": "[#6][SX2H1]",
    "thioether": "[#6][SX2][#6]",
    "sulfone_sulfoxide": "[#16X4](=O)(=O)|[#16X3](=O)",
    "boc": "[NX3]C(=O)OC(C)(C)C",
    "cbz": "[NX3]C(=O)OCc1ccccc1",
    "silyl_ether": "[#8][#14]([#6])([#6])[#6]",
    "phosphorus_ylide_phosphonate": "[#15]",
}

MEDCHEM_SMARTS = {
    k: [Chem.MolFromSmarts(p) for p in v.split("|") if Chem.MolFromSmarts(p) is not None]
    for k, v in RAW_SMARTS.items()
}

def has_fg(mols, key):
    """Checks if any molecule in `mols` matches any SMARTS pattern for `key`."""
    pats = MEDCHEM_SMARTS.get(key, [])
    return any(m is not None and m.HasSubstructMatch(pat) for m in mols for pat in pats)

# --- 3. Local RDKit Heuristic Classifier (with Conditions) ---
# Ordered list of (Condition_Lambda(R, P), Reaction_Name, Typical_Conditions)
RDKIT_REACTION_RULES = [
    # 1. Cross-Couplings
    (lambda R, P: has_fg(R, "boronic") and (has_fg(R, "aryl_halide_pseudohalide") or has_fg(R, "alkyl_lg")),
     "Suzuki-Miyaura Cross-Coupling", "Pd(dppf)Cl2 or Pd(PPh3)4, K2CO3, Dioxane/H2O, 80-100 °C"),
    (lambda R, P: has_fg(R, "boronic") and (has_fg(R, "amine_1_2") or has_fg(R, "phenol")),
     "Chan-Lam Coupling", "Cu(OAc)2, Pyridine or Et3N, DCM, O2 (air), RT"),
    (lambda R, P: has_fg(R, "stannane") and has_fg(R, "aryl_halide_pseudohalide"),
     "Stille Cross-Coupling", "Pd(PPh3)4, LiCl, DMF or Dioxane, 90 °C"),
    (lambda R, P: has_fg(R, "terminal_alkyne") and has_fg(R, "aryl_halide_pseudohalide") and has_fg(P, "internal_alkyne"),
     "Sonogashira Cross-Coupling", "Pd(PPh3)2Cl2, CuI, Et3N, THF/DMF, RT-60 °C"),
    (lambda R, P: has_fg(R, "aryl_halide_pseudohalide") and has_fg(R, "amine_1_2") and has_fg(P, "aryl_amine"),
     "Buchwald-Hartwig Amination / SNAr", "Pd2(dba)3, XPhos/BINAP, NaOtBu, Toluene, 100 °C"),
    (lambda R, P: has_fg(P, "boronic") and has_fg(R, "aryl_halide_pseudohalide") and not has_fg(R, "boronic"),
     "Miyaura Borylation", "B2pin2, Pd(dppf)Cl2, KOAc, Dioxane, 90 °C"),
    (lambda R, P: has_fg(R, "aryl_halide_pseudohalide") and not has_fg(P, "aryl_halide_pseudohalide") and has_fg(R, "alkene") and has_fg(P, "alkene"),
     "Mizoroki-Heck Reaction", "Pd(OAc)2, P(o-tol)3, Et3N, DMF/MeCN, 100 °C"),
    (lambda R, P: has_fg(R, "zinc_magnesium") and has_fg(R, "aryl_halide_pseudohalide"),
     "Kumada / Negishi Cross-Coupling", "Pd(PPh3)4 or Ni(dppe)Cl2, THF, RT-65 °C"),

    # 2. Deprotections
    (lambda R, P: has_fg(R, "boc") and not has_fg(P, "boc"),
     "N-Boc Deprotection (Acidolysis)", "TFA / DCM or 4M HCl in Dioxane, RT"),
    (lambda R, P: has_fg(R, "cbz") and not has_fg(P, "cbz"),
     "N-Cbz Deprotection (Hydrogenolysis)", "H2, Pd/C, MeOH or EtOAc, RT"),
    (lambda R, P: has_fg(R, "silyl_ether") and not has_fg(P, "silyl_ether"),
     "Silyl Ether Deprotection (Desilylation)", "TBAF, THF, 0 °C to RT"),
    (lambda R, P: has_fg(P, "carboxylic_acid") and has_fg(R, "benzyl_ester") and not has_fg(R, "carboxylic_acid"),
     "Benzyl Ester Cleavage (Hydrogenolysis / Hydrolysis)", "H2, Pd/C in MeOH/THF (or LiOH, THF/H2O)"),
    (lambda R, P: has_fg(P, "carboxylic_acid") and has_fg(R, "ester") and not has_fg(R, "carboxylic_acid"),
     "Ester Hydrolysis / Saponification", "LiOH or NaOH, THF / MeOH / H2O, RT-50 °C"),
    (lambda R, P: has_fg(P, "phenol") and has_fg(R, "aryl_ether") and not has_fg(R, "phenol"),
     "Aryl Ether Cleavage / O-Dealkylation", "BBr3, DCM, -78 °C to RT"),

    # 3. Acylations, Amide & Sulfonamide Couplings, Esterifications
    (lambda R, P: has_fg(P, "amide") and not has_fg(R, "amide") and has_fg(R, "amine_1_2") and (has_fg(R, "acyl_halide") or has_fg(R, "anhydride")),
     "Schotten-Baumann Acylation / Amide Coupling", "Et3N or DIPEA, DCM, 0 °C to RT"),
    (lambda R, P: has_fg(P, "amide") and not has_fg(R, "amide") and has_fg(R, "amine_1_2") and has_fg(R, "carboxylic_acid"),
     "Amide Bond Coupling", "HATU or EDC/HOBt, DIPEA, DMF, RT"),
    (lambda R, P: has_fg(P, "sulfonamide") and has_fg(R, "sulfonyl_chloride") and has_fg(R, "amine_1_2"),
     "Sulfonamide Formation", "Et3N or Pyridine, DCM, 0 °C to RT"),
    (lambda R, P: has_fg(P, "ester") and has_fg(R, "carboxylic_acid") and (has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "alkyl_lg")),
     "Esterification (Fischer / Steglich / O-Alkylation)", "EDC, DMAP, DCM (or K2CO3, DMF for alkyl halide)"),

    # 4. Heterocycles & Cycloadditions
    (lambda R, P: has_fg(P, "thiazole") and not has_fg(R, "thiazole") and has_fg(R, "thioamide"),
     "Hantzsch Thiazole Synthesis", "EtOH or DMF, reflux (70-90 °C)"),
    (lambda R, P: has_fg(P, "triazole") and not has_fg(R, "triazole") and has_fg(R, "azide"),
     "CuAAC 'Click' Cycloaddition", "CuSO4, Sodium Ascorbate, t-BuOH/H2O, RT"),
    (lambda R, P: has_fg(P, "tetrazole") and not has_fg(R, "tetrazole") and has_fg(R, "nitrile"),
     "[3+2] Azide-Nitrile Cycloaddition (Tetrazole Synthesis)", "NaN3, NH4Cl or Bu3SnN3, DMF/Toluene, 100-120 °C"),

    # 5. Redox, Substitutions & Functional Group Interconversions
    (lambda R, P: (has_fg(P, "amine_1_2") or has_fg(P, "tert_amine")) and has_fg(R, "aldehyde_ketone") and has_fg(R, "amine_1_2"),
     "Reductive Amination", "NaBH(OAc)3 or NaBH3CN, AcOH, DCE/MeOH, RT"),
    (lambda R, P: has_fg(P, "amine_1_2") and has_fg(R, "nitro") and not has_fg(P, "nitro"),
     "Nitro Reduction to Amine", "H2, Pd/C in MeOH or Fe, NH4Cl in EtOH/H2O, 80 °C"),
    (lambda R, P: has_fg(P, "aryl_ether") and has_fg(R, "phenol") and has_fg(R, "alkyl_lg"),
     "Williamson Ether Synthesis / O-Alkylation", "K2CO3 or Cs2CO3, DMF or MeCN, 60-80 °C"),
    (lambda R, P: has_fg(P, "aryl_ether") and has_fg(R, "phenol") and has_fg(R, "alcohol"),
     "Mitsunobu Etherification", "PPh3, DIAD/DEAD, THF, 0 °C to RT"),
    (lambda R, P: has_fg(P, "nitrile") and not has_fg(R, "nitrile") and has_fg(R, "primary_amide"),
     "Primary Amide Dehydration to Nitrile", "TFAA, Et3N, DCM, 0 °C to RT (or POCl3)"),
    (lambda R, P: has_fg(P, "nitrile") and not has_fg(R, "nitrile") and has_fg(R, "aryl_halide_pseudohalide"),
     "Pd-Catalyzed Aryl Cyanation", "Zn(CN)2, Pd(PPh3)4, DMF, 100 °C"),
]

# --- 4. Enriched Step Classification Schema & Local Chemistry Heuristics ---
REACTION_CLASSIFICATION_CACHE = {}

class StepClassification(BaseModel):
    forward_smiles: str
    reaction_name: str
    typical_conditions: str
    selectivity_warning: str = "None"          # #3: Competing reactive sites or side-reaction risks
    protecting_group_note: str = "None"        # #5: Suggested protection/deprotection choreography
    safety_green_score: int = 4                # #4: 1 (Hazardous/Toxic, e.g., stannanes, azides, DCM) to 5 (Green/Safe)
    greener_solvent_swap: str = "None"         # #4: Recommended green solvent alternative (e.g., EtOAc/2-MeTHF instead of DCM/DMF)
    is_chiral_transformation: bool = False     # #6: True if a stereocenter is formed or inverted
    stereochem_note: str = "Achiral / Racemic" # #6: Asymmetric catalyst suggestion if chiral

class BatchReactionResponse(BaseModel):
    reactions: list[StepClassification]

def _match_rdkit_rule(R, P):
    """Returns (reaction_name, typical_conditions) from RDKIT_REACTION_RULES."""
    for rule_fn, rxn_name, conditions in RDKIT_REACTION_RULES:
        if rule_fn(R, P):
            return rxn_name, conditions
    return "Template-Guided Disconnection", "Standard literature conditions"

def analyze_step_heuristics(forward_smi, rxn_name="", conditions=""):
    """Evaluates chemoselectivity (#3), safety/greenness (#4), protecting groups (#5), and stereochemistry (#6)."""
    defaults = {
        "selectivity_warning": "None",
        "protecting_group_note": "None",
        "safety_green_score": 4,
        "greener_solvent_swap": "None",
        "is_chiral_transformation": False,
        "stereochem_note": "Achiral / Racemic",
    }
    if not forward_smi or ">>" not in forward_smi:
        return defaults

    react_str, prod_str = [s.strip() for s in forward_smi.split(">>", 1)]
    R = [m for s in react_str.split(".") if s and (m := Chem.MolFromSmiles(s)) is not None]
    P = [m for s in prod_str.split(".") if s and (m := Chem.MolFromSmiles(s)) is not None]
    if not R and not P:
        return defaults

    if not rxn_name or not conditions:
        inferred_name, inferred_cond = _match_rdkit_rule(R, P)
        rxn_name = rxn_name or inferred_name
        conditions = conditions or inferred_cond

    rxn_lower = rxn_name.lower()
    cond_lower = conditions.lower()

    # 1. Chemoselectivity (#3) & Protecting Group Choreography (#5)
    warnings = []
    pg_notes = []

    electrophilic_step = (
        has_fg(R, "carboxylic_acid")
        or has_fg(R, "acyl_halide")
        or has_fg(R, "anhydride")
        or has_fg(R, "sulfonyl_chloride")
        or has_fg(R, "alkyl_lg")
        or has_fg(R, "epoxide")
        or any(k in rxn_lower for k in ("acyl", "amide", "sulfonamide", "esterif", "alkylation"))
    )
    if has_fg(R, "amine_1_2") and (has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "thiol")) and electrophilic_step:
        warnings.append(
            "Competing nucleophiles detected (unprotected N-H amine alongside -OH/-SH) — risk of O/S-acylation or over-alkylation."
        )
        pg_notes.append(
            "Consider N-Boc/N-Cbz or O-TBS silyl protection prior to coupling, followed by downstream deprotection."
        )

    is_organometallic_or_coupling = (
        has_fg(R, "boronic")
        or has_fg(R, "stannane")
        or has_fg(R, "zinc_magnesium")
        or any(k in rxn_lower for k in ("suzuki", "stille", "sonogashira", "kumada", "negishi", "heck", "borylation", "cyanation"))
    )
    if is_organometallic_or_coupling:
        if has_fg(R, "zinc_magnesium") and (
            has_fg(R, "aldehyde_ketone") or has_fg(R, "carboxylic_acid") or has_fg(R, "alcohol") or has_fg(R, "phenol") or has_fg(R, "amine_1_2")
        ):
            warnings.append(
                "Unprotected protic or carbonyl group (-OH, -NH, -COOH, C=O) will quench Grignard/organozinc reagent."
            )
            pg_notes.append(
                "Protect carbonyl as acetal, carboxylic acid as ester, and -OH/-NH as O-TBS/N-Boc prior to Kumada/Negishi coupling."
            )
        elif has_fg(R, "thiol"):
            warnings.append(
                "Unprotected thiol (-SH) risks transition-metal catalyst poisoning and competing C-S coupling."
            )
            pg_notes.append("Protect thiol (e.g., S-trityl) prior to Pd/Cu cross-coupling.")
        elif has_fg(R, "aldehyde_ketone") or has_fg(R, "carboxylic_acid") or (
            has_fg(R, "amine_1_2") and "buchwald" not in rxn_lower and "chan-lam" not in rxn_lower
        ):
            warnings.append(
                "Unprotected polar/electrophilic group (free acid, aldehyde/ketone, or N-H amine) may interfere with cross-coupling."
            )
            pg_notes.append(
                "Consider masking carboxylic acid as alkyl/benzyl ester, amine as N-Boc, or carbonyl as acetal prior to coupling."
            )

    halide_sites = sum(
        len(m.GetSubstructMatches(pat))
        for m in R
        for pat in MEDCHEM_SMARTS.get("aryl_halide_pseudohalide", [])
        if hasattr(m, "GetSubstructMatches")
    )
    if halide_sites > 1:
        warnings.append(
            "Multiple aryl halide/pseudohalide sites present — monitor stoichiometry for regioselectivity (Ar-I > Ar-OTf ≈ Ar-Br >> Ar-Cl)."
        )

    # Dedicated deprotection / protecting-group choreography notes (#5)
    if has_fg(R, "boc") and not has_fg(P, "boc"):
        pg_notes.append("N-Boc deprotection step; verify acid-labile groups tolerate TFA/HCl cleavage and pair with upstream Boc2O installation.")
    elif has_fg(R, "cbz") and not has_fg(P, "cbz"):
        pg_notes.append("N-Cbz hydrogenolysis step; ensure alkenes, alkynes, and aryl halides are compatible with H2/Pd-C.")
    elif has_fg(R, "silyl_ether") and not has_fg(P, "silyl_ether"):
        pg_notes.append("O-Silyl deprotection (TBAF); pairs with upstream TBSCl/TBDPSCl hydroxyl protection.")
    elif has_fg(P, "carboxylic_acid") and (has_fg(R, "benzyl_ester") or has_fg(R, "ester")) and not has_fg(R, "carboxylic_acid"):
        pg_notes.append("Late-stage ester cleavage unmasks carboxylic acid cleanly after upstream couplings.")
    elif not pg_notes and (has_fg(R, "boc") or has_fg(R, "cbz") or has_fg(R, "silyl_ether") or has_fg(R, "benzyl_ester")):
        pg_notes.append("Protecting group carried intact through this step; schedule downstream deprotection when unmasking is needed.")

    # 2. Green Chemistry & Lab Safety Scoring (#4)
    safety_score = 4
    swaps = []

    if has_fg(R, "stannane") or has_fg(P, "stannane") or "bu3sn" in cond_lower:
        safety_score = 1
        swaps.append("Replace toxic organotin (Stille) with boronic ester (Suzuki-Miyaura) in 2-MeTHF/H2O or EtOH/H2O")

    if has_fg(R, "azide") or has_fg(P, "azide") or "nan3" in cond_lower:
        safety_score = min(safety_score, 1 if ("nan3" in cond_lower or "100" in cond_lower) else 2)
        swaps.append("Explosion/azide hazard — use flow/blast shield and strictly avoid chlorinated solvents (never mix NaN3 with DCM)")

    if any(tok in cond_lower for tok in ("bbr3", "pocl3", "zn(cn)2", "diad", "dead")):
        safety_score = min(safety_score, 2)

    if has_fg(R, "acyl_halide") or has_fg(R, "sulfonyl_chloride") or "tfaa" in cond_lower or "tfa" in cond_lower:
        safety_score = min(safety_score, 3)

    if "dce" in cond_lower:
        safety_score = min(safety_score, 2)
        swaps.append("EtOAc, 2-MeTHF, or MeOH (replace carcinogenic 1,2-DCE)")
    if "dcm" in cond_lower or "ch2cl2" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("EtOAc, 2-MeTHF, or CPME (replace chlorinated DCM)")
    if "dmf" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("MeCN, EtOAc, DMSO, or Cyrene (replace reprotoxic DMF)")
    if "dioxane" in cond_lower:
        safety_score = min(safety_score, 3)
        swaps.append("2-MeTHF, CPME, or t-AmylOH/H2O (replace carcinogenic 1,4-dioxane)")

    if safety_score == 4 and not swaps and any(g in cond_lower for g in ("etoh", "h2o", "etoac", "meoh", "t-buoh")):
        safety_score = 5

    safety_score = max(1, min(5, int(safety_score)))

    # 3. 3D Stereochemistry / Chiral Center Awareness (#6)
    r_centers = [c for m in R for c in Chem.FindMolChiralCenters(m, includeUnassigned=True)]
    p_centers = [c for m in P for c in Chem.FindMolChiralCenters(m, includeUnassigned=True)]
    r_unassigned = sum(1 for _, lbl in r_centers if lbl == "?")
    p_unassigned = sum(1 for _, lbl in p_centers if lbl == "?")
    r_assigned = sorted(lbl for _, lbl in r_centers if lbl != "?")
    p_assigned = sorted(lbl for _, lbl in p_centers if lbl != "?")

    is_chiral_tx = False
    stereochem_note = "Achiral / Racemic"
    if (
        len(p_centers) > len(r_centers)
        or p_unassigned > r_unassigned
        or (len(p_centers) > 0 and ("mitsunobu" in rxn_lower or p_assigned != r_assigned))
    ):
        is_chiral_tx = True
        if has_fg(R, "aldehyde_ketone"):
            stereochem_note = (
                "New stereocenter from prochiral carbonyl/imine — use Noyori asymmetric hydrogenation, CBS reduction, or chiral transaminase instead of achiral NaBH4/NaBH(OAc)3."
            )
        elif "mitsunobu" in rxn_lower:
            stereochem_note = (
                "SN2 stereochemical inversion at chiral carbinol center — use enantiopure alcohol precursor and monitor enantiomeric excess (ee)."
            )
        elif p_unassigned > 0:
            stereochem_note = (
                "Unassigned stereocenter (?) formed — apply asymmetric catalyst/chiral ligand control or downstream chiral SFC/salt resolution."
            )
        else:
            stereochem_note = (
                "Stereocenter formed or modified — employ chiral catalyst/ligand control or chiral resolution to secure target enantiomer."
            )
    elif len(p_centers) > 0:
        stereochem_note = "Pre-existing stereocenter(s) retained — use mild base/temperature to avoid epimerization."

    return {
        "selectivity_warning": " ".join(warnings) if warnings else "None",
        "protecting_group_note": " ".join(pg_notes) if pg_notes else "None",
        "safety_green_score": safety_score,
        "greener_solvent_swap": "; ".join(dict.fromkeys(swaps)) if swaps else "None",
        "is_chiral_transformation": is_chiral_tx,
        "stereochem_note": stereochem_note,
    }

def _normalize_step_classification(raw_dict):
    """Validates and normalizes a step classification dictionary to all 9 StepClassification fields."""
    obj = StepClassification(**raw_dict)
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    return dict(raw_dict)

def rdkit_classify_step(forward_smi):
    """Evaluates forward reaction SMILES (Reactants >> Product) against RDKit SMARTS rules and local critics."""
    if not forward_smi or ">>" not in forward_smi:
        return _normalize_step_classification({
            "forward_smiles": str(forward_smi or ""),
            "reaction_name": "Functional Group Transformation",
            "typical_conditions": "Standard literature conditions",
        })

    react_str, prod_str = [s.strip() for s in forward_smi.split(">>", 1)]
    R = [Chem.MolFromSmiles(s) for s in react_str.split(".") if s]
    P = [Chem.MolFromSmiles(s) for s in prod_str.split(".") if s]

    rxn_name, conditions = _match_rdkit_rule(R, P)
    heuristics = analyze_step_heuristics(forward_smi, rxn_name=rxn_name, conditions=conditions)
    return _normalize_step_classification({
        "forward_smiles": forward_smi,
        "reaction_name": rxn_name,
        "typical_conditions": conditions,
        **heuristics,
    })

def batch_classify_with_gemini(unique_forward_smiles, use_gemini=True):
    """Classifies uncached reaction steps in a batched Gemini call and updates REACTION_CLASSIFICATION_CACHE with 9-field dicts."""
    if not use_gemini:
        return REACTION_CLASSIFICATION_CACHE

    client = get_gemini_client()
    if client is None:
        return REACTION_CLASSIFICATION_CACHE

    uncached = [s for s in unique_forward_smiles if s not in REACTION_CLASSIFICATION_CACHE]
    if not uncached:
        return REACTION_CLASSIFICATION_CACHE

    prompt = (
        "You are an expert medicinal and process chemist. For each forward reaction SMILES "
        "(Reactants >> Product) below, return a structured StepClassification entry populating ALL 9 fields:\n"
        "1. forward_smiles: exact input SMILES string.\n"
        "2. reaction_name: specific Named Reaction or transformation class (e.g., 'Suzuki-Miyaura Cross-Coupling', 'Hantzsch Thiazole Synthesis').\n"
        "3. typical_conditions: concise, realistic reagents, catalysts, solvents, and temperature.\n"
        "4. selectivity_warning: competing reactive sites or side-reaction risks ('None' if clean).\n"
        "5. protecting_group_note: suggested protection/deprotection choreography ('None' if none needed).\n"
        "6. safety_green_score: integer 1 (hazardous/toxic, e.g., stannanes, azides, DCE) to 5 (green/benign, e.g., EtOH/H2O, EtOAc).\n"
        "7. greener_solvent_swap: greener solvent alternative such as EtOAc, 2-MeTHF, or CPME instead of DCM/DMF/Dioxane ('None' if already green).\n"
        "8. is_chiral_transformation: boolean True if a stereocenter is formed, unassigned, or inverted.\n"
        "9. stereochem_note: asymmetric catalyst/chiral resolution recommendation if chiral, else 'Achiral / Racemic'.\n\n"
        + "\n".join(f"- {smi}" for smi in uncached)
    )

    response = None
    last_err = None
    for model_name in ("gemini-3.8-flash", "gemini-2.5-flash"):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": BatchReactionResponse,
                    "temperature": 0.1,
                },
            )
            break
        except Exception as e:
            last_err = e

    if response is None:
        print(f"ℹ️ Gemini batch classification skipped ({last_err}); falling back to local RDKit SMARTS.")
        return REACTION_CLASSIFICATION_CACHE

    try:
        parsed = json.loads(response.text)
        for item in parsed.get("reactions", []):
            fwd_smi = item.get("forward_smiles", "")
            if not fwd_smi:
                continue
            local_fallback = rdkit_classify_step(fwd_smi)
            merged = dict(local_fallback)
            for k, v in item.items():
                if v is not None and v != "":
                    merged[k] = v
            merged["safety_green_score"] = max(1, min(5, int(merged.get("safety_green_score", 4))))
            REACTION_CLASSIFICATION_CACHE[fwd_smi] = _normalize_step_classification(merged)
    except Exception as e:
        print(f"ℹ️ Gemini batch classification parse skipped ({e}); falling back to local RDKit SMARTS.")

    return REACTION_CLASSIFICATION_CACHE

def to_forward_smiles(retro_smi):
    """Converts AiZynthFinder's Product>>Reactants string into Reactants >> Product."""
    if ">>" in retro_smi:
        prod_s, react_s = retro_smi.split(">>", 1)
        return f"{react_s.strip()} >> {prod_s.strip()}"
    return retro_smi
# --- 5. RDKit MCP Native Chemistry, 2D SVG Depiction & LC-MS Helpers ---
import html as _chem_html
from rdkit.Chem import Descriptors, rdMolDescriptors, rdChemReactions
from rdkit.Chem.Draw import rdMolDraw2D
_CHEM_SVG_CARDS = set()


def _chem_mol(smiles):
    if not smiles or len(smiles) > 4096:
        return None
    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
    except (ValueError, RuntimeError):
        return None
    if mol is None or Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
        return None
    return mol


def mol_or_rxn_to_svg_html(smiles_or_rxn, width=260, height=150, highlight_smarts="", label=""):
    """Draw a validated molecule or reaction as an inline SVG card."""
    parts = smiles_or_rxn.split(">>")
    if len(parts) not in (1, 2):
        return "Invalid molecule/reaction SMILES."
    sides = [[_chem_mol(s.strip()) for s in side.split(".") if s.strip()] for side in parts]
    if any(not side or len(side) > 16 or any(m is None for m in side) for side in sides):
        return "Invalid molecule/reaction SMILES."
    try:
        drawer = rdMolDraw2D.MolDraw2DSVG(max(100, min(int(width), 1200)), max(80, min(int(height), 800)))
        drawer.drawOptions().prepareMolsBeforeDrawing = False
        def prepared(mol):
            try:
                return rdMolDraw2D.PrepareMolForDrawing(mol, kekulize=False)
            except Exception:
                return Chem.Mol(mol)
        if len(sides) == 2:
            rxn = rdChemReactions.ChemicalReaction()
            for mol in sides[0]:
                rxn.AddReactantTemplate(prepared(mol))
            for mol in sides[1]:
                rxn.AddProductTemplate(prepared(mol))
            drawer.DrawReaction(rxn)
        else:
            mol = prepared(sides[0][0])
            pat = Chem.MolFromSmarts(highlight_smarts) if highlight_smarts else None
            atoms = mol.GetSubstructMatch(pat) if pat is not None else ()
            drawer.DrawMolecule(mol, highlightAtoms=list(atoms))
        drawer.FinishDrawing()
        svg = drawer.GetDrawingText().replace("<?xml version='1.0' encoding='iso-8859-1'?>", "")
        card = f'<div class="chem-card">{_chem_html.escape(label)}{svg.replace(chr(10), " ")}</div>'
        if len(_CHEM_SVG_CARDS) >= 1000:
            _CHEM_SVG_CARDS.clear()
        _CHEM_SVG_CARDS.add(card)
        return card
    except Exception:
        return "Unable to depict validated structure."


def rdkit_enumerate_byproducts(forward_smi):
    """Enumerate plausible competing products, not a quantitative prediction."""
    if forward_smi.count(">>") != 1:
        return []
    left, right = forward_smi.split(">>")
    reactants = [_chem_mol(s.strip()) for s in left.split(".") if s.strip()]
    products = [_chem_mol(s.strip()) for s in right.split(".") if s.strip()]
    if not reactants or not products or len(reactants) > 16 or len(products) > 16 or any(m is None for m in reactants + products):
        return []
    seen = set()
    results = []
    def add(mol, origin):
        if mol is None or Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
            return
        smi = Chem.MolToSmiles(mol)
        if smi in seen or len(results) >= 24:
            return
        seen.add(smi)
        results.append({"origin": origin, "smiles": smi, "formula": rdMolDescriptors.CalcMolFormula(mol),
                        "exact_mw": Descriptors.ExactMolWt(mol), "svg_html": mol_or_rxn_to_svg_html(smi, label=origin)})
    for mol in products:
        add(mol, "target product")
    rules = []
    if has_fg(reactants, "amine_1_2") and (has_fg(reactants, "alcohol") or has_fg(reactants, "phenol") or has_fg(reactants, "thiol")) and (has_fg(reactants, "acyl_halide") or has_fg(reactants, "carboxylic_acid")):
        for nucleophile, label in (("[O;H1:3]", "O-acylation"), ("[S;H1:3]", "S-acylation")):
            for leaving in ("[Cl,Br]", "[O;H1]"):
                rules.append((f"[C:1](=[O:2]){leaving}.{nucleophile}[#6:5]>>[C:1](=[O:2])[#8,#16:3][#6:5]", label, 2))
    if has_fg(reactants, "boronic") and has_fg(reactants, "aryl_halide_pseudohalide"):
        rules.extend([
            ("[c:1][B]([O])[O]>>[cH:1]", "protodeboronation", 1),
            ("[c:1][B]([O])[O].[c:5][B]([O])[O]>>[c:1]-[c:5]", "boronic homocoupling", 2),
            ("[c:1][Cl,Br,I].[c:3][B]([O])[O]>>[c:1]-[c:3]", "halide regioisomer", 2),
        ])
    if has_fg(reactants, "ester"):
        rules.append(("[C:1](=[O:2])[O:3][#6]>>[C:1](=[O:2])[OH:3]", "ester cleavage", 1))
    for smarts, label, arity in rules:
        try:
            rxn = rdChemReactions.ReactionFromSmarts(smarts)
            if rxn is None:
                continue
            for a in reactants:
                for b in (reactants if arity == 2 else [None]):
                    for outcome in rxn.RunReactants((a, b) if arity == 2 else (a,), maxProducts=24):
                        for product in outcome:
                            add(product, label)
        except (ValueError, RuntimeError):
            continue
    return results


def rdkit_predict_lcms_ions(forward_smi):
    """Return exact monoisotopic m/z values for reaction monitoring (singly charged)."""
    if forward_smi.count(">>") != 1:
        return []
    left, right = forward_smi.split(">>")
    entries = [("reactant", s.strip()) for s in left.split(".") if s.strip()]
    entries += [("target product", s.strip()) for s in right.split(".") if s.strip()]
    entries += [(p["origin"], p["smiles"]) for p in rdkit_enumerate_byproducts(forward_smi) if p["origin"] != "target product"]
    ions = []
    for role, smi in dict.fromkeys(entries):
        mol = _chem_mol(smi)
        if mol is None:
            continue
        mass = Descriptors.ExactMolWt(mol)
        cl = sum(a.GetAtomicNum() == 17 for a in mol.GetAtoms())
        br = sum(a.GetAtomicNum() == 35 for a in mol.GetAtoms())
        ions.append({"role": role, "smiles": smi, "formula": rdMolDescriptors.CalcMolFormula(mol), "exact_mw": mass,
                     "[M+H]+": mass + 1.007276466621, "[M+Na]+": mass + 22.989218,
                     "[M+NH4]+": mass + 18.033823, "[M+MeCN+H]+": mass + 42.033825,
                     "[M-H]-": mass - 1.007276466621,
                     "isotope_note": f"M/M+2: Cl ~3:1 per atom (x{cl}); Br ~1:1 per atom (x{br})" if (cl or br) else "No Cl/Br M+2 signature"})
    return ions


def rdkit_predict_workup_properties(forward_smi):
    """Suggest extraction and chromatography from product properties and reactant contrast."""
    if forward_smi.count(">>") != 1:
        return {"error": "Expected reactants >> product SMILES"}
    left, right = forward_smi.split(">>")
    reactants = [_chem_mol(s.strip()) for s in left.split(".") if s.strip()]
    products = [_chem_mol(s.strip()) for s in right.split(".") if s.strip()]
    if not products or any(m is None for m in reactants + products):
        return {"error": "Invalid molecule SMILES"}
    def props(m):
        return {"smiles": Chem.MolToSmiles(m), "clogp": Descriptors.MolLogP(m), "tpsa": Descriptors.TPSA(m),
                "hbd": rdMolDescriptors.CalcNumHBD(m), "hba": rdMolDescriptors.CalcNumHBA(m),
                "acid": has_fg([m], "carboxylic_acid"), "base": has_fg([m], "amine_1_2")}
    product = props(products[0])
    precursors = [props(m) for m in reactants]
    advice = []
    if product["clogp"] < 0.5 or product["tpsa"] > 110:
        advice.append("Aqueous extraction may lose polar product (cLogP < 0.5 or TPSA > 110 Å²); check both phases by LC-MS, salt out or use reverse-phase purification.")
    if product["acid"]:
        advice.append("Extract acid with aqueous NaHCO3; acidify carefully before re-extraction (CO2 evolution).")
    elif product["base"]:
        advice.append("Extract amine with 1M HCl or citric acid, then basify before re-extraction; avoid acid-sensitive groups.")
    else:
        advice.append("Use a neutral water/brine wash; confirm product partitioning by LC-MS.")
    delta_logp = min((abs(product["clogp"] - p["clogp"]) for p in precursors), default=0)
    delta_tpsa = min((abs(product["tpsa"] - p["tpsa"]) for p in precursors), default=0)
    advice.append("Reverse-phase chromatography recommended for polar or closely co-eluting species." if product["tpsa"] > 110 or product["clogp"] < 0.5 or (delta_logp < 0.7 and delta_tpsa < 20) else "Normal-phase silica chromatography is a reasonable starting point; screen by TLC/LC-MS.")
    return {"product": product, "reactants": precursors, "delta_clogp": delta_logp, "delta_tpsa": delta_tpsa, "advice": advice}


In [ ]:
# @title 3. Initialize AiZynthFinder Engine
import os
import csv
import logging
import tempfile
from rdkit import Chem
from aizynthfinder.aizynthfinder import AiZynthFinder
from aizynthfinder.context.stock import InMemoryInchiKeyQuery

# Silence verbose DEBUG/INFO output across the notebook
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)
for handler in logging.getLogger("aizynthfinder").handlers:
    handler.setLevel(logging.WARNING)

CUSTOM_LAB_STOCK_INCHIKEYS = set()
CUSTOM_LAB_STOCK_SMILES = set()

def register_custom_lab_stock(smiles_list, stock_name="lab_inventory"):
    """Validates SMILES via RDKit, builds an InMemoryInchiKeyQuery stock, and selects ['zinc', stock_name]."""
    if isinstance(smiles_list, str):
        candidates = [tok.strip() for line in smiles_list.splitlines() for tok in line.split(",") if tok.strip()]
    else:
        candidates = [str(s).strip() for s in (smiles_list or []) if str(s).strip()]

    header_tokens = {"smiles", "canonical_smiles", "smi", "inchikey", "inchi_key", "structure", "compound"}
    valid_inchikeys = []
    valid_smiles = []
    seen_keys = set()
    invalid_count = 0

    for item in candidates:
        if item.lower() in header_tokens:
            continue
        mol = Chem.MolFromSmiles(item)
        if mol is None or mol.GetNumAtoms() == 0:
            invalid_count += 1
            continue
        canon_smi = Chem.MolToSmiles(mol)
        inchi_key = Chem.MolToInchiKey(mol)
        if inchi_key and inchi_key not in seen_keys:
            seen_keys.add(inchi_key)
            valid_inchikeys.append(inchi_key)
            valid_smiles.append(canon_smi)

    if not valid_inchikeys:
        raise ValueError("No valid chemical SMILES found to load into custom lab stock.")

    stock_dir = "data" if os.path.isdir("data") else tempfile.gettempdir()
    os.makedirs(stock_dir, exist_ok=True)
    stock_csv_path = os.path.join(stock_dir, f"{stock_name}_inchikeys.csv")
    with open(stock_csv_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["inchi_key"])
        for ik in valid_inchikeys:
            writer.writerow([ik])

    stock_query = InMemoryInchiKeyQuery(stock_csv_path)
    CUSTOM_LAB_STOCK_INCHIKEYS.update(valid_inchikeys)
    CUSTOM_LAB_STOCK_SMILES.update(valid_smiles)

    selected_stocks = ["zinc", stock_name]
    if "finder" in globals() and finder is not None:
        finder.stock.load(stock_query, stock_name)
        finder.stock.select(selected_stocks)
        finder.custom_lab_stock_inchikeys = set(CUSTOM_LAB_STOCK_INCHIKEYS)
        finder.custom_lab_stock_smiles = set(CUSTOM_LAB_STOCK_SMILES)

    print(f"📦 Loaded {len(valid_inchikeys)} unique building blocks into '{stock_name}' (active stocks: {selected_stocks}).")
    return {
        "stock_name": stock_name,
        "loaded_count": len(valid_inchikeys),
        "invalid_count": invalid_count,
        "selected_stocks": selected_stocks,
        "inchikeys": valid_inchikeys,
    }

if "finder" not in globals():
    finder = AiZynthFinder(configfile="faster-ml-casp/configs/medusa_default_config.yml")
    finder.stock.select("zinc")
    finder.expansion_policy.select("transformer")
    finder.config.search.algorithm = "aizynthfinder.search.retrostar.search_tree.SearchTree"
    finder.config.search.return_first = True
    finder.config.post_processing.min_routes = 1
    finder.config.post_processing.max_routes = 10
    print("✅ AiZynthFinder initialized with GPU Medusa (M-SBS), Retro* search, and ZINC stock.")
else:
    print("✅ AiZynthFinder instance already active in memory.")

In [ ]:
# @title 4. Interactive Target Molecule Sketcher (JSME) + Multi-Identifier Resolver
import re
import urllib.parse
import requests
import IPython
from google.colab import output
from rdkit import Chem
from rdkit.Chem import Draw

# 1. Chemical Identifier Resolver (SMILES, InChI, InChIKey, PubChem CID, DrugBank, ChEMBL, CAS, Name)
def _fetch_pubchem_smiles(namespace, identifier):
    """Queries PubChem PUG-REST for Canonical/Connectivity SMILES."""
    encoded = urllib.parse.quote(str(identifier).strip())
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/{namespace}/{encoded}/property/CanonicalSMILES,ConnectivitySMILES/JSON"
    resp = requests.get(url, timeout=8)
    if resp.status_code == 200:
        props = resp.json().get("PropertyTable", {}).get("Properties", [])
        if props:
            p = props[0]
            return p.get("CanonicalSMILES") or p.get("ConnectivitySMILES") or p.get("SMILES")
    return None

def _resolve_chemical_identifier(query_str):
    """Resolves SMILES, InChI, InChIKey, PubChem CID, DrugBank ID, ChEMBL ID, CAS, or Drug Name."""
    q = (query_str or "").strip()
    if not q:
        return None, "Empty input"

    # A. Local RDKit Check: InChI string
    if q.startswith("InChI="):
        mol = Chem.MolFromInchi(q)
        if mol:
            return Chem.MolToSmiles(mol), "InChI (RDKit)"

    # B. Local RDKit Check: Direct SMILES (skip if it looks like a pure number/CID or word name)
    if not q.isdigit() and not re.match(r"^(DB\d+|CHEMBL\d+|[A-Z]{14}-[A-Z]{10}-[A-Z])$", q, re.I):
        mol = Chem.MolFromSmiles(q)
        if mol and mol.GetNumAtoms() > 0:
            return Chem.MolToSmiles(mol), "SMILES (RDKit)"

    # C. InChIKey (27 chars: 14-10-1)
    if re.match(r"^[A-Z]{14}-[A-Z]{10}-[A-Z]$", q, re.I):
        smi = _fetch_pubchem_smiles("inchikey", q.upper())
        if smi:
            return smi, f"InChIKey ({q.upper()})"

    # D. PubChem CID (handles '54732242', 'CID:54732242', 'PubChem CID 54732242', etc.)
    cid_match = re.match(r"^(?:pubchem[\s_-]*)?(?:cid[:=\s_-]*)?(\d+)$", q, re.I)
    if cid_match:
        cid = cid_match.group(1)
        smi = _fetch_pubchem_smiles("cid", cid)
        if smi:
            return smi, f"PubChem CID ({cid})"

    # E. ChEMBL ID (e.g., CHEMBL1234)
    if re.match(r"^CHEMBL\d+$", q, re.I):
        chembl_id = q.upper()
        try:
            r = requests.get(f"https://www.ebi.ac.uk/chembl/api/data/molecule/{chembl_id}.json", timeout=8)
            if r.status_code == 200:
                structs = r.json().get("molecule_structures") or {}
                smi = structs.get("canonical_smiles")
                if smi:
                    return smi, f"ChEMBL ({chembl_id})"
        except Exception:
            pass
        smi = _fetch_pubchem_smiles("name", chembl_id)
        if smi:
            return smi, f"ChEMBL via PubChem ({chembl_id})"

    # F. DrugBank ID (e.g., DB04868) -> PubChem Registry / Name / UniChem fallback
    if re.match(r"^DB\d{5}$", q, re.I):
        db_id = q.upper()
        smi = _fetch_pubchem_smiles("xref/RegistryID", db_id) or _fetch_pubchem_smiles("name", db_id)
        if smi:
            return smi, f"DrugBank ({db_id})"
        # Fallback via EBI UniChem API (DrugBank source ID = 2 -> ChEMBL source ID = 1)
        try:
            u_resp = requests.get(f"https://www.ebi.ac.uk/unichem/rest/src_compound_id/{db_id}/2/1", timeout=8)
            if u_resp.status_code == 200 and u_resp.json():
                chembl_hit = u_resp.json()[0].get("src_compound_id")
                if chembl_hit:
                    return _resolve_chemical_identifier(chembl_hit)
        except Exception:
            pass

    # G. Generic Drug Name, Synonym, or CAS Number -> PubChem Name Lookup + NCI CACTUS fallback
    smi = _fetch_pubchem_smiles("name", q)
    if smi:
        return smi, f"PubChem Name/Registry ('{q}')"

    try:
        cactus_url = f"https://cactus.nci.nih.gov/chemical/structure/{urllib.parse.quote(q)}/smiles"
        c_resp = requests.get(cactus_url, timeout=8)
        if c_resp.status_code == 200 and c_resp.text.strip():
            first_line = c_resp.text.strip().splitlines()[0].strip()
            if Chem.MolFromSmiles(first_line):
                return first_line, f"NCI CACTUS ('{q}')"
    except Exception:
        pass

    return None, f"Could not resolve identifier: '{q}'"

# 2. Colab Callbacks for Lookup, Atom-Locking & Custom Lab Stock
def _resolve_frozen_atoms(raw_mol, canon_mol, frozen_indices=None):
    """Maps JSME atom-starred maps, atom index lists, or SMARTS substructures onto canonical 0-based atom indices."""
    locked = []
    locked_smarts = None
    n_atoms = canon_mol.GetNumAtoms() if canon_mol is not None else 0

    # A. Detect starred/atom-mapped atoms directly from JSME SMILES if present
    if raw_mol is not None and canon_mol is not None and hasattr(raw_mol, "GetAtoms"):
        starred_raw = [a.GetIdx() for a in raw_mol.GetAtoms() if a.GetAtomMapNum() > 0]
        if starred_raw:
            unmapped_copy = Chem.Mol(raw_mol)
            for a in unmapped_copy.GetAtoms():
                a.SetAtomMapNum(0)
            match = canon_mol.GetSubstructMatch(unmapped_copy)
            if match:
                locked.extend(match[i] for i in starred_raw if i < len(match))

    # B. Parse explicit frozen_indices (list/tuple of ints, comma-separated ints, or SMARTS/SMILES pattern)
    if isinstance(frozen_indices, (list, tuple, set)):
        raw_ints = [int(x) for x in frozen_indices]
        if raw_ints and all(0 <= i < n_atoms for i in raw_ints):
            locked.extend(raw_ints)
        elif raw_ints and all(1 <= i <= n_atoms for i in raw_ints):
            locked.extend(i - 1 for i in raw_ints)
    elif isinstance(frozen_indices, str) and frozen_indices.strip():
        spec = frozen_indices.strip()
        if re.match(r"^[\d,\s]+$", spec):
            raw_ints = [int(tok) for tok in re.split(r"[,\s]+", spec) if tok.strip()]
            if raw_ints and all(0 <= i < n_atoms for i in raw_ints):
                locked.extend(raw_ints)
            elif raw_ints and all(1 <= i <= n_atoms for i in raw_ints):
                locked.extend(i - 1 for i in raw_ints)
            else:
                locked.extend(i for i in raw_ints if 0 <= i < n_atoms)
        else:
            pat = Chem.MolFromSmarts(spec)
            if pat is None:
                pat = Chem.MolFromSmiles(spec)
            if pat is not None and canon_mol is not None:
                match = canon_mol.GetSubstructMatch(pat)
                if match:
                    locked.extend(match)
                    locked_smarts = spec

    return sorted(set(locked)), locked_smarts

def _lookup_and_set_target(query_str):
    smi, source_label = _resolve_chemical_identifier(query_str)
    if not smi:
        return IPython.display.JSON({"error": source_label})

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return IPython.display.JSON({"error": f"Resolved invalid SMILES from {source_label}: {smi}"})

    canon_smiles = Chem.MolToSmiles(mol)
    if "finder" in globals():
        finder.target_smiles = canon_smiles
        finder.CustomFrozenAtoms = []
        finder.CustomFrozenSmarts = None

    return IPython.display.JSON({
        "smiles": canon_smiles,
        "source": source_label,
    })

def _set_target_smiles(smiles_str, frozen_indices=None):
    smiles_str = (smiles_str or "").strip()
    raw_mol = Chem.MolFromSmiles(smiles_str)
    if raw_mol is None or not smiles_str:
        print(f"⚠️ Invalid or empty structure: '{smiles_str}'")
        return

    clean_mol = Chem.Mol(raw_mol)
    if hasattr(clean_mol, "GetAtoms"):
        for atom in clean_mol.GetAtoms():
            if atom.GetAtomMapNum() > 0:
                atom.SetAtomMapNum(0)

    canon_smiles = Chem.MolToSmiles(clean_mol)
    canon_mol = Chem.MolFromSmiles(canon_smiles) or clean_mol
    locked_atoms, locked_smarts = _resolve_frozen_atoms(raw_mol, canon_mol, frozen_indices=frozen_indices)

    if "finder" in globals():
        finder.target_smiles = canon_smiles
        finder.CustomFrozenAtoms = locked_atoms
        finder.CustomFrozenSmarts = locked_smarts

    if locked_atoms:
        print(f"✅ Successfully set finder.target_smiles = '{canon_smiles}' | 🔒 Locked atom indices: {locked_atoms}")
        display(Draw.MolToImage(canon_mol, size=(320, 200), highlightAtoms=locked_atoms))
    else:
        print(f"✅ Successfully set finder.target_smiles = '{canon_smiles}'")
        display(Draw.MolToImage(canon_mol, size=(320, 200)))

def _load_custom_stock_callback(raw_text, stock_name="lab_inventory"):
    text = (raw_text or "").strip()
    if not text:
        return IPython.display.JSON({"error": "Please paste or upload at least one SMILES string."})

    parsed_smiles = []
    header_skip = {"smiles", "canonical_smiles", "smi", "name", "id", "catalog_id", "amount", "inchikey", "inchi_key"}
    for line in text.splitlines():
        row = line.strip()
        if not row or row.startswith("#"):
            continue
        cols = [c.strip().strip('"').strip("'") for c in re.split(r"[,;\t]", row) if c.strip()]
        if not cols:
            continue
        if cols[0].lower() in header_skip and len(cols) >= 1:
            continue
        chosen = None
        for col in cols:
            if col.lower() in header_skip:
                continue
            m = Chem.MolFromSmiles(col)
            if m is not None and m.GetNumAtoms() > 0:
                chosen = col
                break
        parsed_smiles.append(chosen if chosen is not None else cols[0])

    try:
        info = register_custom_lab_stock(parsed_smiles, stock_name=stock_name or "lab_inventory")
        return IPython.display.JSON(info)
    except Exception as e:
        return IPython.display.JSON({"error": str(e)})

output.register_callback("notebook.resolve_identifier", _lookup_and_set_target)
output.register_callback("notebook.set_target_smiles", _set_target_smiles)
output.register_callback("notebook.load_custom_stock", _load_custom_stock_callback)

# 3. Render Interactive Identifier Search + JSME Molecule Sketcher + Constraints Drawer
html_code = """
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; width: 680px;">
  <div style="margin-bottom: 6px; font-weight: 600; font-size: 14px; color: #202124;">
    🔎 1. Search by Chemical Identifier (Drug Name, PubChem CID, DrugBank ID, ChEMBL ID, InChIKey, CAS, or SMILES):
  </div>
  <div style="display: flex; gap: 8px; margin-bottom: 6px;">
    <input id="chem-id-input" type="text"
           placeholder="e.g., Febuxostat | DB04868 | 134018 | CHEMBL175691 | BAXJSDGLLature... | CAS 144060-53-7"
           style="flex: 1; padding: 8px 10px; font-size: 13px; border: 1px solid #ccc; border-radius: 5px;" />
    <button id="lookup-id-btn"
            style="background-color: #1a73e8; color: white; border: none; padding: 8px 14px;
                   font-size: 13px; font-weight: 600; border-radius: 5px; cursor: pointer; white-space: nowrap;">
      🔍 Fetch Structure
    </button>
  </div>
  <div style="display: flex; gap: 6px; flex-wrap: wrap; align-items: center; margin-bottom: 10px; font-size: 12px; color: #5f6368;">
    <span>Quick examples:</span>
    <button onclick="quickLookup('Febuxostat')" style="border:1px solid #dadce0; background:#f8f9fa; border-radius:12px; padding:2px 9px; cursor:pointer; font-size:11px;">Name: Febuxostat</button>
    <button onclick="quickLookup('DB04868')" style="border:1px solid #dadce0; background:#f8f9fa; border-radius:12px; padding:2px 9px; cursor:pointer; font-size:11px;">DrugBank: DB04868 (Nilotinib)</button>
    <button onclick="quickLookup('5479530')" style="border:1px solid #dadce0; background:#f8f9fa; padding:2px 9px; border-radius:12px; cursor:pointer; font-size:11px;">PubChem CID: 5479530 (Naldemedine)</button>
    <button onclick="quickLookup('KTUFNOKKBVMGRW-UHFFFAOYSA-N')" style="border:1px solid #dadce0; background:#f8f9fa; padding:2px 9px; border-radius:12px; cursor:pointer; font-size:11px;">InChIKey: Imatinib</button>
  </div>

  <div style="margin-bottom: 6px; font-weight: 600; font-size: 14px; color: #202124;">
    ✏️ 2. Or Draw / Edit Structure Directly in the Sketcher (use ★ tool to pin/lock atoms):
  </div>
  <div id="jsme_container" style="width: 680px; height: 400px; border: 1px solid #ccc; border-radius: 6px;"></div>

  <div style="margin-top: 10px; display: flex; align-items: center; gap: 10px; width: 680px;">
    <input id="manual-smiles" type="text" placeholder="Active SMILES string..."
           style="flex: 1; padding: 8px; font-family: monospace; font-size: 13px; border: 1px solid #ccc; border-radius: 4px;"
           value="Cc1nc(C2=CC=C(OCC(C)C)C(C#N)=C2)sc1C(=O)O" />
    <button id="save-smiles-btn" style="background-color: #28a745; color: white; border: none;
            padding: 9px 16px; font-size: 13px; font-weight: bold; border-radius: 5px; cursor: pointer; white-space: nowrap;">
      ✅ Set as finder.target_smiles
    </button>
  </div>
  <div id="smiles-status" style="margin-top: 6px; font-family: monospace; font-size: 13px; color: #155724;"></div>

  <details style="margin-top: 10px; border: 1px solid #dadce0; border-radius: 6px; padding: 8px 12px; background: #f8f9fa;">
    <summary style="cursor: pointer; font-weight: 600; font-size: 13px; color: #202124;">
      ⚙️ 3. Optional Constraints: Substructure Atom-Locking &amp; Custom Lab Inventory Stock
    </summary>
    <div style="margin-top: 10px; display: flex; flex-direction: column; gap: 10px; font-size: 12px;">
      <div>
        <label style="font-weight: 600; color: #3c4043; display: block; margin-bottom: 4px;">
          🔒 Frozen Atom Indices / SMARTS Substructure to Keep Intact:
        </label>
        <input id="frozen-atoms-input" type="text"
               placeholder="Lock core ring SMARTS or atom indices (e.g., c1ncsc1 or 0,1,2,3)"
               style="width: 100%; box-sizing: border-box; padding: 6px 8px; font-family: monospace; font-size: 12px; border: 1px solid #ccc; border-radius: 4px;" />
      </div>
      <div>
        <label style="font-weight: 600; color: #3c4043; display: block; margin-bottom: 4px;">
          🧪 Custom Lab Inventory SMILES ("What's in My Lab Freezer?" CSV / List):
        </label>
        <textarea id="custom-stock-input" rows="3"
                  placeholder="Paste one SMILES per line or upload a CSV of in-house building blocks..."
                  style="width: 100%; box-sizing: border-box; padding: 6px 8px; font-family: monospace; font-size: 12px; border: 1px solid #ccc; border-radius: 4px;"></textarea>
        <div style="margin-top: 6px; display: flex; align-items: center; gap: 8px; flex-wrap: wrap;">
          <input id="custom-stock-file" type="file" accept=".csv,.txt,.smi" style="font-size: 12px;" />
          <button id="load-stock-btn" type="button"
                  style="background-color: #1a73e8; color: white; border: none; padding: 6px 12px; font-size: 12px; font-weight: 600; border-radius: 4px; cursor: pointer;">
            📦 Load Lab Stock
          </button>
          <span id="custom-stock-status" style="font-family: monospace; font-size: 12px; color: #155724;"></span>
        </div>
      </div>
    </div>
  </details>
</div>

<script>
function initJSME() {
  if (typeof JSApplet !== "undefined") {
    window.jsmeApplet = new JSApplet.JSME("jsme_container", "680px", "400px", {"options": "star"});
    const defaultSmiles = document.getElementById("manual-smiles").value;
    window.jsmeApplet.readGenericMolecularInput(defaultSmiles);
    window.jsmeApplet.setCallBack("AfterStructureModified", function(event) {
      const s = window.jsmeApplet.smiles();
      if (s) document.getElementById("manual-smiles").value = s;
    });
  }
}

window.jsmeOnLoad = function() { initJSME(); };

if (!document.getElementById("jsme-script")) {
  const script = document.createElement("script");
  script.id = "jsme-script";
  script.src = "https://jsme-editor.github.io/dist/jsme/jsme.nocache.js";
  document.head.appendChild(script);
} else {
  setTimeout(initJSME, 300);
}

async function resolveIdentifier() {
  const query = document.getElementById("chem-id-input").value.trim();
  const statusEl = document.getElementById("smiles-status");
  const lookupBtn = document.getElementById("lookup-id-btn");
  if (!query) return;

  statusEl.style.color = "#1a73e8";
  statusEl.innerText = "⏳ Resolving '" + query + "' across RDKit, PubChem, ChEMBL, DrugBank & CACTUS...";
  lookupBtn.disabled = true;

  try {
    const res = await google.colab.kernel.invokeFunction("notebook.resolve_identifier", [query], {});
    const data = res.data["application/json"];
    if (data.error) {
      statusEl.style.color = "#d93025";
      statusEl.innerText = "⚠️ " + data.error;
    } else {
      document.getElementById("manual-smiles").value = data.smiles;
      if (window.jsmeApplet) {
        window.jsmeApplet.readGenericMolecularInput(data.smiles);
      }
      statusEl.style.color = "#155724";
      statusEl.innerText = "✅ Resolved via " + data.source + " → finder.target_smiles = '" + data.smiles + "'";
    }
  } catch (err) {
    statusEl.style.color = "#d93025";
    statusEl.innerText = "⚠️ Kernel communication error: " + err;
  } finally {
    lookupBtn.disabled = false;
  }
}

function quickLookup(val) {
  document.getElementById("chem-id-input").value = val;
  resolveIdentifier();
}

document.getElementById("lookup-id-btn").onclick = resolveIdentifier;
document.getElementById("chem-id-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter") {
    e.preventDefault();
    resolveIdentifier();
  }
});

// Sync manual SMILES input box changes to JSME canvas on Enter or blur
document.getElementById("manual-smiles").addEventListener("change", function(e) {
  const smi = e.target.value.trim();
  if (smi && window.jsmeApplet) {
    window.jsmeApplet.readGenericMolecularInput(smi);
  }
});

document.getElementById("save-smiles-btn").onclick = function() {
  const statusEl = document.getElementById("smiles-status");
  const manualInput = document.getElementById("manual-smiles").value.trim();
  const jsmeSmiles = (window.jsmeApplet && window.jsmeApplet.smiles()) ? window.jsmeApplet.smiles().trim() : "";
  const frozenSpec = document.getElementById("frozen-atoms-input").value.trim();

  // Prefer JSME atom-mapped SMILES if user starred atoms on canvas, otherwise manual input
  const smiles = (jsmeSmiles && jsmeSmiles.indexOf(":") !== -1) ? jsmeSmiles : (manualInput || jsmeSmiles);
  if (window.jsmeApplet && smiles) {
    window.jsmeApplet.readGenericMolecularInput(smiles);
  }
  statusEl.style.color = "#155724";
  statusEl.innerText = "Setting finder.target_smiles = " + smiles + (frozenSpec ? " (lock: " + frozenSpec + ")" : "") + " ...";
  google.colab.kernel.invokeFunction("notebook.set_target_smiles", [smiles, frozenSpec], {});
};

document.getElementById("custom-stock-file").addEventListener("change", function(e) {
  const file = e.target.files && e.target.files[0];
  if (!file) return;
  const reader = new FileReader();
  reader.onload = function(evt) {
    document.getElementById("custom-stock-input").value = evt.target.result || "";
  };
  reader.readAsText(file);
});

document.getElementById("load-stock-btn").onclick = async function() {
  const rawText = document.getElementById("custom-stock-input").value.trim();
  const stockStatus = document.getElementById("custom-stock-status");
  const loadBtn = document.getElementById("load-stock-btn");
  if (!rawText) {
    stockStatus.style.color = "#d93025";
    stockStatus.innerText = "⚠️ Paste SMILES or select a CSV file first.";
    return;
  }
  loadBtn.disabled = true;
  stockStatus.style.color = "#1a73e8";
  stockStatus.innerText = "⏳ Loading custom lab stock...";
  try {
    const res = await google.colab.kernel.invokeFunction("notebook.load_custom_stock", [rawText], {});
    const data = res.data["application/json"];
    if (data.error) {
      stockStatus.style.color = "#d93025";
      stockStatus.innerText = "⚠️ " + data.error;
    } else {
      stockStatus.style.color = "#155724";
      stockStatus.innerText = "✅ Loaded " + data.loaded_count + " compounds into '" + data.stock_name + "'";
    }
  } catch (err) {
    stockStatus.style.color = "#d93025";
    stockStatus.innerText = "⚠️ Kernel error: " + err;
  } finally {
    loadBtn.disabled = false;
  }
};
</script>
"""
IPython.display.display(IPython.display.HTML(html_code))

In [ ]:
# @title 5. Configure Search & Route Ranking Settings { display-mode: "form" }
# @markdown ### ⚙️ Interactive AiZynthFinder & Route Ranking Settings
# @markdown Adjust the text boxes below; run Cell 5B to launch the search.

# @markdown ---
# @markdown **1. Route Ranking Profile (`ranking_mode`)**

# @markdown *How to adjust:* Select a preset strategy from thedropdown.

# @markdown *Modes:*

# @markdown **`balanced`** weighs step count, stock availability, and feasibility equally;

# @markdown **`medchem_fast:`** prioritizes the shortest routes for rapid analog generation;

# @markdown **`process_scaleup:`** penalizes hazardous reagents and low-yielding steps;

# @markdown **`green_safe:`** prioritizes benign solvents and high safety scores.

ranking_mode = "green_safe" # @param ["balanced", "medchem_fast", "process_scaleup", "green_safe"]


# @markdown **2. Maximum Synthetic Steps (`max_transforms`)**

# @markdown *How to adjust:* Slide between 1 and 12 steps (default: `6`).

# @markdown *Effect:* Caps the depth of the retrosynthetic tree. Lower values (3–5) finish faster and force shorter routes; higher values (7–10) allow deeper disconnections for complex scaffolds like Naldemedine.
max_transforms = 6 

# @markdown **3. Search Time Limit in Seconds (`time_limit`)**

# @markdown *How to adjust:* Slide between 15 and 600 seconds (default: `120`).

# @markdown *Effect:* Sets the maximum wall-clock time for the Monte Carlo Tree Search (MCTS). Increasing this gives AiZynthFinder more time to resolve complex intermediates into purchasable stock.
time_limit = 120 

# @markdown **4. MCTS Iteration Limit (`iteration_limit`)**

# @markdown *How to adjust:* Slide between 10 and 500 iterations (default: `100`).

# @markdown *Effect:* Controls how many tree expansion/rollout cycles are performed. Higher values explore more diverse synthetic pathways but take longer to run.
iteration_limit = 220 

# @markdown **5. Stop at First Solved Route (`return_first`)**

# @markdown *How to adjust:* Check or uncheck the box (default: `False`).

# @markdown *Effect:* When checked (`True`), the search halts the moment it finds a single route where all starting materials are in stock—great for a fast 5-second feasibility check, but yields fewer route alternatives.
return_first = False # @param {type:"boolean"}

# @markdown **6. Template Cumulative Probability Cutoff (`cutoff_cumulative`)**

# @markdown *How to adjust:* Enter a decimal between `0.500` and `1.000` (default: `0.995`).

# @markdown *Effect:* Controls how many reaction templates the neural network keeps at each step. Lowering to `0.90`–`0.95` prunes unlikely templates for a faster, more conservative search; raising toward `0.999` explores rarer, more creative disconnections.
cutoff_cumulative = 0.995 # @param {type:"number"}

# @markdown **7. Maximum Top Routes to Extract (`max_routes_to_extract`)**

# @markdown *How to adjust:* Slide between 1 and 25 routes (default: `10`).

# @markdown *Effect:* Sets the maximum number of top-ranked reaction trees extracted, classified, and displayed after the search finishes.
max_routes_to_extract = 10 

# @markdown **8. Target Synthesis Scale in Grams (`target_scale_grams`)**

# @markdown *How to adjust:* Enter the desired final product mass in grams (default: `1.0`).

# @markdown *Effect:* Scales the stoichiometric mass estimates and flags scale-up hazards (e.g., azides, organotins, or cryogenic steps) more strictly at larger scales.
target_scale_grams = 1.0 # @param {type:"number"}

# @markdown **9. Exclude Specific Reaction Types (`exclude_reactions_csv`)**

# @markdown *How to adjust:* Type comma-separated reaction keywords to ban, or leave blank (e.g., `Stille, Azide, Mitsunobu`).

# @markdown *Effect:* Filters out or heavily penalizes routes containing any of the listed reaction names so you can avoid reagents you don't want to run in the lab.
exclude_reactions_csv = "" # @param {type:"string"}

# @markdown **10. Enable Gemini AI Chemist Classification (`use_gemini_classifier`)**

# @markdown *How to adjust:* Check (`True`) to use Gemini API + RDKit, or uncheck (`False`) for local RDKit-only rules.

# @markdown *Effect:* When enabled, sends unique reaction steps to Gemini to generate named reactions, realistic conditions, protecting group choreography, and greener solvent swaps.
use_gemini_classifier = True # @param {type:"boolean"}

import html as _html
import logging
import markdown as _md
import numpy as np
from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem import Descriptors

# Keep AiZynthFinder DEBUG spam suppressed during search
logging.getLogger("aizynthfinder").setLevel(logging.WARNING)

try:
    from paretoset import paretoset as _paretoset_fn
except Exception:
    _paretoset_fn = None

USE_GEMINI_CLASSIFIER = bool(use_gemini_classifier)
LATEST_RETRO_RESULTS = {}

def _compute_pareto_mask(obj_matrix):
    """Computes Pareto-optimal boolean mask (maximizing all columns in obj_matrix)."""
    n_rows = len(obj_matrix)
    if n_rows == 0:
        return []
    if _paretoset_fn is not None:
        arr = np.asarray(obj_matrix, dtype=float)
        mask = _paretoset_fn(arr, sense=["max"] * arr.shape[1], distinct=False)
        return [bool(x) for x in mask]
    # Pure-Python dominance fallback
    mask = []
    for i, row_i in enumerate(obj_matrix):
        dominated = False
        for j, row_j in enumerate(obj_matrix):
            if i == j:
                continue
            if all(b >= a for a, b in zip(row_i, row_j)) and any(b > a for a, b in zip(row_i, row_j)):
                dominated = True
                break
        mask.append(not dominated)
    return mask

def evaluate_starting_materials(rxn_tree, check_pubchem_vendors=False):
    """Inspects leaf starting materials of rxn_tree for custom lab stock, ZINC stock, and cost/complexity tier (#2)."""
    if hasattr(rxn_tree, "leafs"):
        leaf_nodes = list(rxn_tree.leafs())
    elif hasattr(rxn_tree, "molecules"):
        all_mols = list(rxn_tree.molecules())
        if hasattr(rxn_tree, "graph"):
            leaf_nodes = [m for m in all_mols if not rxn_tree.graph[m]]
        else:
            leaf_nodes = all_mols
    else:
        leaf_nodes = []

    lab_keys = set()
    lab_smis = set()
    if "finder" in globals() and finder is not None:
        lab_keys.update(getattr(finder, "custom_lab_stock_inchikeys", set()) or set())
        lab_smis.update(getattr(finder, "custom_lab_stock_smiles", set()) or set())
    lab_keys.update(globals().get("CUSTOM_LAB_STOCK_INCHIKEYS", set()) or set())
    lab_smis.update(globals().get("CUSTOM_LAB_STOCK_SMILES", set()) or set())

    precursors = []
    for leaf in leaf_nodes:
        smi = getattr(leaf, "smiles", str(leaf))
        rd_mol = getattr(leaf, "rd_mol", None) or Chem.MolFromSmiles(smi)
        inchi_key = getattr(leaf, "inchi_key", None) or (
            Chem.MolToInchiKey(rd_mol) if rd_mol is not None and hasattr(Chem, "MolToInchiKey") else ""
        )

        in_lab = bool((inchi_key and inchi_key in lab_keys) or (smi and smi in lab_smis))
        if not in_lab and "finder" in globals() and finder is not None and hasattr(finder, "stock"):
            try:
                stock_items = getattr(finder.stock, "items", None) or getattr(finder.stock, "_items", {})
                if "lab_inventory" in stock_items and leaf in finder.stock["lab_inventory"]:
                    in_lab = True
            except Exception:
                pass

        if callable(getattr(rxn_tree, "in_stock", None)):
            tree_in_stock = bool(rxn_tree.in_stock(leaf))
        else:
            tree_in_stock = bool(getattr(leaf, "in_stock", False))
        in_stock = in_lab or tree_in_stock

        heavy_atoms = (
            rd_mol.GetNumHeavyAtoms()
            if rd_mol is not None and hasattr(rd_mol, "GetNumHeavyAtoms")
            else max(1, len(smi) // 2)
        )
        chiral_cnt = (
            len(Chem.FindMolChiralCenters(rd_mol, includeUnassigned=True))
            if rd_mol is not None and hasattr(Chem, "FindMolChiralCenters")
            else 0
        )
        specialty_atoms = (
            sum(1 for a in rd_mol.GetAtoms() if a.GetAtomicNum() in {5, 14, 15, 34, 50, 53})
            if rd_mol is not None and hasattr(rd_mol, "GetAtoms")
            else 0
        )

        if in_lab:
            source = "Lab Inventory (On-Shelf)"
            cost_tier = "$0 On-Shelf"
            est_usd = 0.0
        elif in_stock:
            source = "ZINC Stock"
            if heavy_atoms <= 12 and chiral_cnt == 0 and specialty_atoms == 0:
                cost_tier = "$ Low"
                est_usd = 25.0
            elif heavy_atoms <= 22 and chiral_cnt <= 1 and specialty_atoms <= 1:
                cost_tier = "$$ Medium"
                est_usd = 120.0
            else:
                cost_tier = "$$$ Expensive"
                est_usd = 450.0
        else:
            source = "Requires Synthesis"
            cost_tier = "$$$ Expensive"
            est_usd = 1000.0

        vendor_count = None
        if check_pubchem_vendors and smi:
            try:
                import urllib.parse
                import requests
                enc = urllib.parse.quote(smi)
                r = requests.get(
                    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{enc}/cids/JSON",
                    timeout=5,
                )
                if r.status_code == 200:
                    cids = r.json().get("IdentifierList", {}).get("CID", [])
                    if cids:
                        vr = requests.get(
                            f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cids[0]}/JSON?heading=Chemical+Vendors",
                            timeout=5,
                        )
                        if vr.status_code == 200:
                            vendor_count = vr.text.count('"SourceName"')
            except Exception:
                vendor_count = None

        precursors.append({
            "smiles": smi,
            "inchi_key": inchi_key,
            "in_stock": in_stock,
            "in_lab_inventory": in_lab,
            "source": source,
            "cost_tier": cost_tier,
            "estimated_usd_per_g": est_usd,
            "heavy_atoms": heavy_atoms,
            "chiral_centers": chiral_cnt,
            "vendor_count": vendor_count,
        })

    total_leaves = len(precursors)
    in_stock_count = sum(1 for p in precursors if p["in_stock"])
    lab_count = sum(1 for p in precursors if p["in_lab_inventory"])
    stock_coverage = (in_stock_count / total_leaves) if total_leaves else 1.0
    total_est_usd = sum(p["estimated_usd_per_g" ] for p in precursors)

    if total_leaves > 0 and lab_count == total_leaves:
        overall_tier = "$0 On-Shelf"
    elif in_stock_count < total_leaves:
        overall_tier = "$$$ Expensive (Unsolved)"
    elif total_est_usd <= 80.0:
        overall_tier = "$ Low"
    elif total_est_usd <= 300.0:
        overall_tier = "$$ Medium"
    else:
        overall_tier = "$$$ Expensive"

    return {
        "precursors": precursors,
        "total_leaves": total_leaves,
        "in_stock_count": in_stock_count,
        "lab_inventory_count": lab_count,
        "stock_coverage": round(stock_coverage, 4),
        "estimated_total_usd_per_g": round(total_est_usd, 2),
        "overall_cost_tier": overall_tier,
    }

def compute_composite_route_score(metrics, ranking_mode="balanced"):
    """Computes multi-objective score combining state score, step count, safety, selectivity, stock, and cost (#9)."""
    mode = (ranking_mode or "balanced").strip().lower()
    state_score = float(metrics.get("state_score", 0.0))
    step_count = int(metrics.get("step_count", 1))
    step_eff = 1.0 / (1.0 + 0.25 * max(0, step_count - 1))
    safety_norm = float(metrics.get("avg_safety_score", 4.0)) / 5.0
    sel_clean = max(0.0, 1.0 - 0.30 * int(metrics.get("selectivity_risk_count", 0)))
    stock_cov = float(metrics.get("stock_coverage", 1.0))
    cost_eff = max(0.0, 1.0 - min(float(metrics.get("estimated_total_usd_per_g", 100.0)), 1000.0) / 1000.0)
    policy_prob = float(metrics.get("avg_policy_prob", state_score))

    if mode == "medchem_fast":
        score = 0.38 * step_eff + 0.28 * state_score + 0.20 * stock_cov + 0.09 * policy_prob + 0.05 * safety_norm
    elif mode == "process_scaleup":
        score = 0.25 * safety_norm + 0.25 * sel_clean + 0.20 * cost_eff + 0.15 * stock_cov + 0.15 * policy_prob
    elif mode == "green_safe":
        score = 0.50 * safety_norm + 0.20 * sel_clean + 0.15 * stock_cov + 0.15 * state_score
    else:
        score = 0.30 * state_score + 0.20 * step_eff + 0.20 * safety_norm + 0.15 * sel_clean + 0.15 * stock_cov
    return round(float(score), 4)

def _route_preserves_substructure(steps, lock_pat):
    """Returns True if every step whose product contains lock_pat retains lock_pat in at least one reactant."""
    if lock_pat is None:
        return True
    for st in steps:
        fwd = st.get("forward_smiles", "")
        if ">>" not in fwd:
            continue
        react_s, prod_s = [x.strip() for x in fwd.split(">>", 1)]
        prod_mols = [Chem.MolFromSmiles(s) for s in prod_s.split(".") if s]
        if not any(m is not None and m.HasSubstructMatch(lock_pat) for m in prod_mols):
            continue
        react_mols = [Chem.MolFromSmiles(s) for s in react_s.split(".") if s]
        if not any(m is not None and m.HasSubstructMatch(lock_pat) for m in react_mols):
            return False
    return True

def _route_matches_exclusion(steps, excluded_terms):
    """Returns True if any step in the route matches a banned reaction/catalyst keyword."""
    if not excluded_terms:
        return False
    pd_aliases = ("pd(", "pd2(", "pd/", "suzuki", "stille", "sonogashira", "buchwald", "heck", "miyaura")
    for st in steps:
        haystack = f"{st.get('reaction_name', '')} {st.get('typical_conditions', '')}".lower()
        for term in excluded_terms:
            if term in haystack:
                return True
            if term in ("palladium", "pd", "pd catalyst") and any(a in haystack for a in pd_aliases):
                return True
    return False

def generate_eln_markdown(route_idx=1, target_scale_grams=1.0, assumed_step_yield=0.80, results=None):
    """Generates a scaled stoichiometry table and step-by-step ELN lab recipe for a route (#10)."""
    active = results if results is not None else globals().get("LATEST_RETRO_RESULTS")
    if not active or not active.get("routes"):
        return "⚠️ No retrosynthesis routes available in memory. Please run Cell 5 first."

    routes_list = active["routes"]
    idx0 = int(route_idx) - 1
    if idx0 < 0 or idx0 >= len(routes_list):
        return f"⚠️ Route #{route_idx} is out of range (1..{len(routes_list)} available)."

    route_entry = routes_list[idx0]
    target_smi = active.get("target_smiles", "")
    target_mol = _chem_mol(target_smi) if target_smi else None
    target_mw = float(Descriptors.ExactMolWt(target_mol)) if target_mol is not None else 250.0
    if target_mw <= 0:
        target_mw = 250.0

    scale_g = max(0.001, float(target_scale_grams))
    step_yield = min(1.0, max(0.05, float(assumed_step_yield)))
    final_target_mmol = (scale_g * 1000.0) / target_mw

    steps = route_entry.get("steps", [])
    n_steps = len(steps)
    overall_yield = (step_yield ** n_steps) if n_steps > 0 else 1.0

    md_lines = [
        f"#### 🧪 Electronic Lab Notebook (ELN) Protocol — Route #{route_idx}",
        f"- **Target SMILES:** `{target_smi}` (Exact MW: `{target_mw:.2f} g/mol`)",
        f"- **Target Delivery Scale:** `{scale_g:.2f} g` (`{final_target_mmol:.2f} mmol`) | **Assumed Yield/Step:** `{step_yield * 100:.0f}%` (Overall `{overall_yield * 100:.1f}%` over {n_steps} step{'s' if n_steps != 1 else ''})",
        f"- **Composite Score:** `{route_entry.get('composite_score', 0.0):.4f}` | **Safety/Green Rating:** `{route_entry.get('avg_safety_score', 4.0):.1f}/5` | **Precursor Tier:** `{route_entry.get('starting_materials', {}).get('overall_cost_tier', 'N/A')}`",
        "",
        "##### ⚖️ Scaled Stoichiometry Table",
        "| Step | Role | SMILES | 2D Structure | Exact MW (g/mol) | Equiv | Required (mmol) | Mass (g) | Mass (mg) |",
        "| :---: | :--- | :--- | :--- | ---: | ---: | ---: | ---: | ---: |",
    ]

    proc_lines = ["", "##### 📝 Step-by-Step Procedure & Workup Checklist"]
    for s_i, st in enumerate(steps, start=1):
        steps_remaining = int(st.get("steps_to_target") or (n_steps - s_i + 1))
        expected_prod_mmol = final_target_mmol / (step_yield ** max(0, steps_remaining - 1))
        limiting_mmol = expected_prod_mmol / step_yield

        fwd_smi = st.get("forward_smiles", "")
        react_part, prod_part = [p.strip() for p in fwd_smi.split(">>", 1)] if ">>" in fwd_smi else ("", "")
        react_smis = [s.strip() for s in react_part.split(".") if s.strip()]
        prod_smis = [s.strip() for s in prod_part.split(".") if s.strip()]

        for r_idx, r_smi in enumerate(react_smis):
            r_mol = _chem_mol(r_smi)
            r_mw = float(Descriptors.ExactMolWt(r_mol)) if r_mol is not None else 150.0
            equiv = 1.00 if r_idx == 0 else 1.05
            req_mmol = limiting_mmol * equiv
            mass_mg = req_mmol * r_mw
            mass_g = mass_mg / 1000.0
            role = "Limiting Reactant" if r_idx == 0 else f"Co-Reactant {r_idx + 1}"
            md_lines.append(
                f"| {s_i} | {role} | `{r_smi}` | {mol_or_rxn_to_svg_html(r_smi, width=150, height=95)} | {r_mw:.2f} | {equiv:.2f} | {req_mmol:.2f} | {mass_g:.3f} | {mass_mg:.1f} |"
            )

        if prod_smis:
            p_mol = _chem_mol(prod_smis[0])
            p_mw = float(Descriptors.ExactMolWt(p_mol)) if p_mol is not None else target_mw
            p_mg = expected_prod_mmol * p_mw
            p_g = p_mg / 1000.0
            md_lines.append(
                f"| {s_i} | Expected Product ({int(step_yield * 100)}%) | `{prod_smis[0]}` | {mol_or_rxn_to_svg_html(prod_smis[0], width=150, height=95)} | {p_mw:.2f} | 1.00 | {expected_prod_mmol:.2f} | {p_g:.3f} | {p_mg:.1f} |"
            )

        proc_lines.append(
            f"{s_i}. **{st.get('reaction_name', 'Reaction Step')}** — Charge limiting reactant (`{limiting_mmol:.2f} mmol`, `1.00 eq`) and coupling partner(s) under **{st.get('typical_conditions', 'standard conditions')}**."
        )
        if st.get("greener_solvent_swap", "None") != "None":
            proc_lines.append(f"   - 🌱 *Green Solvent Recommendation:* {st['greener_solvent_swap']}")
        if st.get("selectivity_warning", "None") != "None":
            proc_lines.append(f"   - ⚠️ *Selectivity Check:* {st['selectivity_warning']}")
        if st.get("protecting_group_note", "None") != "None":
            proc_lines.append(f"   - 🔒 *Protecting Group Choreography:* {st['protecting_group_note']}")
        if st.get("stereochem_note", "Achiral / Racemic") != "Achiral / Racemic":
            proc_lines.append(f"   - 🧬 *Stereochemistry:* {st['stereochem_note']}")
        workup = rdkit_predict_workup_properties(fwd_smi)
        if "advice" in workup:
            p = workup["product"]
            proc_lines.append(f"   - *Workup:* cLogP {p['clogp']:.2f}, TPSA {p['tpsa']:.1f} Å², HBD/HBA {p['hbd']}/{p['hba']}; " + " ".join(workup["advice"]))
        ions = rdkit_predict_lcms_ions(fwd_smi)
        if ions:
            proc_lines.append("   - *LC-MS Reaction Monitoring (m/z):* " + "; ".join(
                f"{item['role']} `{item['smiles']}` " + ", ".join(
                    f"{ion} {item[ion]:.4f}" for ion in ("[M+H]+", "[M+Na]+", "[M+NH4]+", "[M+MeCN+H]+", "[M-H]-")
                ) + f" ({item['isotope_note']})"
                for item in ions
            ))

    return "\n".join(md_lines + proc_lines)

def run_retrosynthesis_pipeline(
    target_smiles: str = None,
    ranking_mode: str = "balanced",  # "medchem_fast" | "process_scaleup" | "green_safe" | "balanced"
    max_transforms: int = 6,
    time_limit: int = 120,
    exclude_reaction_names: list[str] = None,
    lock_substructure_smarts: str = None,
    target_scale_grams: float = 1.0,
    use_gemini: bool = True,
    render_output: bool = True,
    max_routes_to_extract: int = 10,
) -> dict:
    """Executes constrained AiZynthFinder search, chemical critic scoring, Pareto ranking, and ELN generation."""
    global LATEST_RETRO_RESULTS, gemini_lookup

    active_target = (target_smiles or getattr(finder, "target_smiles", "") or "").strip()
    if not active_target:
        raise ValueError("⚠️ Please set a target molecule in Cell 4 before running the search.")

    target_mol_rd = Chem.MolFromSmiles(active_target)
    if target_mol_rd is not None:
        active_target = Chem.MolToSmiles(target_mol_rd)
        target_mol_rd = Chem.MolFromSmiles(active_target)

    finder.target_smiles = active_target
    if hasattr(finder, "config") and hasattr(finder.config, "search"):
        finder.config.search.max_transforms = int(max_transforms)
        finder.config.search.time_limit = int(time_limit)

    # Apply substructure / atom-locking constraints (#8)
    effective_lock_smarts = (
        (lock_substructure_smarts or getattr(finder, "CustomFrozenSmarts", "") or "").strip() or None
    )
    lock_pat = None
    locked_atom_indices = list(getattr(finder, "CustomFrozenAtoms", []) or [])

    if effective_lock_smarts and target_mol_rd is not None:
        lock_pat = Chem.MolFromSmarts(effective_lock_smarts) or Chem.MolFromSmiles(effective_lock_smarts)
        if lock_pat is not None:
            match = target_mol_rd.GetSubstructMatch(lock_pat)
            if match:
                locked_atom_indices = list(match)
                finder.CustomFrozenAtoms = locked_atom_indices
    elif locked_atom_indices and target_mol_rd is not None and hasattr(Chem, "MolFragmentToSmiles"):
        try:
            frag_smi = Chem.MolFragmentToSmiles(target_mol_rd, atomsToUse=locked_atom_indices)
            if frag_smi:
                lock_pat = Chem.MolFromSmarts(frag_smi) or Chem.MolFromSmiles(frag_smi)
                effective_lock_smarts = frag_smi
        except Exception:
            pass

    if hasattr(finder, "filter_policy"):
        try:
            if effective_lock_smarts:
                from aizynthfinder.context.policy import FrozenSubstructureFilter
                sub_filter = FrozenSubstructureFilter(
                    "__frozen_substruct", finder.config, smarts_list=[effective_lock_smarts]
                )
                finder.filter_policy.load(sub_filter)
                finder.filter_policy.select("__frozen_substruct", append=True)
            elif "__frozen_substruct" in (getattr(finder.filter_policy, "selection", None) or []):
                finder.filter_policy.deselect("__frozen_substruct")
        except Exception:
            pass

    if hasattr(finder, "prepare_tree"):
        finder.prepare_tree()

    # 1. Run MCTS tree search and extract candidate routes
    finder.tree_search()
    finder.build_routes()
    stats = finder.extract_statistics() or {}

    candidate_routes = list(finder.routes)
    unique_steps = list(dict.fromkeys(
        to_forward_smiles(rxn_node.reaction_smiles())
        for route in candidate_routes
        for rxn_node in route["reaction_tree"].reactions()
        if ">>" in rxn_node.reaction_smiles()
    ))

    gemini_lookup = batch_classify_with_gemini(unique_steps, use_gemini=use_gemini)
    excluded_terms = [str(t).strip().lower() for t in (exclude_reaction_names or []) if str(t).strip()]

    enriched_routes = []
    filtered_out_count = 0

    for route in candidate_routes:
        state_score = float(route.get("score", {}).get("state score", 0.0))
        rxn_tree = route["reaction_tree"]
        reactions_forward = list(reversed(list(rxn_tree.reactions())))

        steps_data = []
        policy_probs = []
        for s_idx, rxn_node in enumerate(reactions_forward, start=1):
            meta = getattr(rxn_node, "metadata", {}) or {}
            template_code = meta.get("template_code", meta.get("template_hash", "N/A"))
            policy_prob = meta.get("policy_probability", None)
            if isinstance(policy_prob, (int, float)):
                policy_probs.append(float(policy_prob))

            fwd_smi = to_forward_smiles(rxn_node.reaction_smiles())
            cached_info = gemini_lookup.get(fwd_smi) if isinstance(gemini_lookup, dict) else None
            if isinstance(cached_info, dict):
                step_info = dict(cached_info)
            elif isinstance(cached_info, (tuple, list)) and len(cached_info) >= 2:
                step_info = rdkit_classify_step(fwd_smi)
                step_info["reaction_name"] = cached_info[0]
                step_info["typical_conditions"] = cached_info[1]
            else:
                step_info = rdkit_classify_step(fwd_smi)

            depth_val = rxn_tree.depth(rxn_node) if callable(getattr(rxn_tree, "depth", None)) else -1
            steps_to_target = (depth_val // 2) + 1 if isinstance(depth_val, int) and depth_val >= 1 else (
                len(reactions_forward) - s_idx + 1
            )

            step_entry = dict(step_info)
            step_entry.update({
                "step_index": s_idx,
                "template_code": template_code,
                "policy_probability": policy_prob,
                "steps_to_target": steps_to_target,
            })
            steps_data.append(step_entry)

        if lock_pat is not None and not _route_preserves_substructure(steps_data, lock_pat):
            filtered_out_count += 1
            continue

        if _route_matches_exclusion(steps_data, excluded_terms):
            filtered_out_count += 1
            continue

        sm_eval = evaluate_starting_materials(rxn_tree)
        step_count = len(steps_data)
        avg_safety = (
            sum(float(s.get("safety_green_score", 4)) for s in steps_data) / step_count
            if step_count > 0
            else 5.0
        )
        selectivity_risks = sum(1 for s in steps_data if s.get("selectivity_warning", "None") != "None")
        avg_prob = (sum(policy_probs) / len(policy_probs)) if policy_probs else state_score

        metrics = {
            "state_score": state_score,
            "step_count": step_count,
            "avg_safety_score": round(avg_safety, 2),
            "selectivity_risk_count": selectivity_risks,
            "stock_coverage": sm_eval["stock_coverage"],
            "estimated_total_usd_per_g": sm_eval["estimated_total_usd_per_g"],
            "avg_policy_prob": round(avg_prob, 4),
        }
        comp_score = compute_composite_route_score(metrics, ranking_mode=ranking_mode)

        enriched_routes.append({
            "raw_route": route,
            "reaction_tree": rxn_tree,
            "is_solved": bool(getattr(rxn_tree, "is_solved", False)),
            "state_score": round(state_score, 4),
            "composite_score": comp_score,
            "step_count": step_count,
            "avg_safety_score": metrics["avg_safety_score"],
            "selectivity_risk_count": selectivity_risks,
            "stock_coverage": sm_eval["stock_coverage"],
            "starting_materials": sm_eval,
            "steps": steps_data,
            "is_pareto_optimal": False,
        })

    # Compute Pareto-optimal frontier before slicing
    if enriched_routes:
        obj_matrix = [
            [
                r["state_score"],
                -float(r["step_count"]),
                r["avg_safety_score"],
                -float(r["selectivity_risk_count"]),
                r["stock_coverage"],
            ]
            for r in enriched_routes
        ]
        pareto_flags = _compute_pareto_mask(obj_matrix)
        for r_entry, is_opt in zip(enriched_routes, pareto_flags):
            r_entry["is_pareto_optimal"] = is_opt

    enriched_routes.sort(
        key=lambda r: (r["composite_score"], r["is_pareto_optimal"], r["state_score"]),
        reverse=True,
    )
    top_routes = enriched_routes[:int(max_routes_to_extract)]

    LATEST_RETRO_RESULTS = {
        "target_smiles": active_target,
        "ranking_mode": ranking_mode,
        "max_transforms": max_transforms,
        "time_limit": time_limit,
        "excluded_reactions": excluded_terms,
        "locked_substructure_smarts": effective_lock_smarts,
        "locked_atom_indices": locked_atom_indices,
        "target_scale_grams": target_scale_grams,
        "is_solved": bool(stats.get("is_solved", any(r["is_solved"] for r in top_routes))),
        "total_routes_found": len(candidate_routes),
        "filtered_out_routes": filtered_out_count,
        "routes": top_routes,
    }

    if render_output:
        classifier_mode = "Gemini Structured + RDKit Critics" if (use_gemini and gemini_lookup) else "Local RDKit MedChem Critics"
        lock_msg = f" | Locked Atoms: {locked_atom_indices}" if locked_atom_indices else ""
        excl_msg = f" | Excluded: {excluded_terms}" if excluded_terms else ""
        print(f"Target SMILES: {active_target} | Mode: {ranking_mode} | Classifier: {classifier_mode}{lock_msg}{excl_msg}")
        print(
            f"Search Solved: {LATEST_RETRO_RESULTS['is_solved']} | Displaying Top {len(top_routes)} of "
            f"{len(candidate_routes)} Routes"
            + (f" ({filtered_out_count} filtered by constraints)" if filtered_out_count else "")
            + "\n" + "=" * 95
        )

        for idx, r_entry in enumerate(top_routes, start=1):
            rxn_tree = r_entry["reaction_tree"]
            sm_info = r_entry["starting_materials"]

            badges = []
            if r_entry["is_pareto_optimal"]:
                badges.append(
                    "<span style='background:#fef7e0; color:#b06000; border:1px solid #fbbc04; padding:2px 8px; border-radius:12px; font-size:12px;'>🏆 Pareto Optimal</span>"
                )
            safety_val = r_entry["avg_safety_score"]
            s_bg, s_fg = ("#e6f4ea", "#137333") if safety_val >= 3.8 else (("#fef7e0", "#b06000") if safety_val >= 2.5 else ("#fce8e6", "#c5221f"))
            badges.append(
                f"<span style='background:{s_bg}; color:{s_fg}; padding:2px 8px; border-radius:12px; font-size:12px;'>🛡️ Safety/Green: {safety_val:.1f}/5</span>"
            )
            badges.append(
                f"<span style='background:#e8f0fe; color:#174ea6; padding:2px 8px; border-radius:12px; font-size:12px;'>💰 Precursors: {_html.escape(sm_info['overall_cost_tier'])} ({sm_info['in_stock_count']}/{sm_info['total_leaves']} in stock)</span>"
            )
            if r_entry["selectivity_risk_count"] > 0:
                badges.append(
                    f"<span style='background:#fce8e6; color:#c5221f; padding:2px 8px; border-radius:12px; font-size:12px;'>⚠️ {r_entry['selectivity_risk_count']} Selectivity Risk(s)</span>"
                )
            else:
                badges.append(
                    "<span style='background:#e6f4ea; color:#137333; padding:2px 8px; border-radius:12px; font-size:12px;'>✅ Clean Selectivity</span>"
                )
            if any(s.get("protecting_group_note", "None") != "None" for s in r_entry["steps"]):
                badges.append(
                    "<span style='background:#f3e8fd; color:#681da8; padding:2px 8px; border-radius:12px; font-size:12px;'>🔒 PG Choreography</span>"
                )
            if any(s.get("is_chiral_transformation") for s in r_entry["steps"]):
                badges.append(
                    "<span style='background:#e4f7fb; color:#007b83; padding:2px 8px; border-radius:12px; font-size:12px;'>🧬 Chiral Control</span>"
                )

            header_html = (
                f"<div style='margin-top:12px; margin-bottom:6px; font-family:-apple-system,BlinkMacSystemFont,sans-serif;'>"
                f"<h3 style='margin:0 0 6px 0; color:#1a73e8;'>Route #{idx} &mdash; Composite: {r_entry['composite_score']:.4f} "
                f"| State Score: {r_entry['state_score']:.4f} (Solved: {r_entry['is_solved']})</h3>"
                f"<div style='display:flex; gap:6px; flex-wrap:wrap;'>{''.join(badges)}</div></div>"
            )
            display(HTML(header_html))
            if hasattr(rxn_tree, "to_image"):
                display(rxn_tree.to_image())

            n_st = r_entry["step_count"]
            print(f"Chemical Reaction Steps ({n_st} step{'s' if n_st != 1 else ''}, Left-to-Right Forward Order):\n")
            for st in r_entry["steps"]:
                prob = st.get("policy_probability")
                prob_str = f" | Policy Prob: {prob:.3f}" if isinstance(prob, (int, float)) else ""
                print(
                    f"\033[1m  Step {st['step_index']}: {st['reaction_name']}  "
                    f"[Template ID: {st['template_code']}{prob_str} | Safety: {st['safety_green_score']}/5]\033[0m"
                )
                print(f"          Suggested Conditions: {st['typical_conditions']}")
                print(f"          Forward SMILES:       {st['forward_smiles']}")
                if st.get("greener_solvent_swap", "None") != "None":
                    print(f"          🌱 Greener Swap:      {st['greener_solvent_swap']}")
                if st.get("selectivity_warning", "None") != "None":
                    print(f"          ⚠️ Selectivity Note:  {st['selectivity_warning']}")
                if st.get("protecting_group_note", "None") != "None":
                    print(f"          🔒 Protecting Group:  {st['protecting_group_note']}")
                if st.get("stereochem_note", "Achiral / Racemic") != "Achiral / Racemic":
                    print(f"          🧬 Stereochemistry:   {st['stereochem_note']}")
                print()  # Empty line between steps

            eln_md = generate_eln_markdown(
                idx, target_scale_grams=target_scale_grams, results=LATEST_RETRO_RESULTS
            )
            try:
                eln_body_html = _md.markdown(eln_md, extensions=["tables", "fenced_code"])
            except Exception:
                eln_body_html = f"<pre>{_html.escape(eln_md)}</pre>"

            eln_details_html = (
                "<details style='margin:8px 0 14px 0; border:1px solid #dadce0; border-radius:6px; padding:8px 12px; "
                "background:#fcfcfc; font-family:-apple-system,BlinkMacSystemFont,sans-serif; font-size:13px;'>"
                f"<summary style='cursor:pointer; font-weight:600; color:#1a73e8;'>📋 View {target_scale_grams:.1f}g Stoichiometry &amp; ELN Protocol</summary>"
                f"<div style='margin-top:8px;'>{eln_body_html}</div></details>"
            )
            display(HTML(eln_details_html))
            print("-" * 95)

    return LATEST_RETRO_RESULTS

# --- Sync Interactive Form Parameters to AiZynthFinder Config ---
if "finder" in globals() and finder is not None and hasattr(finder, "config"):
    finder.config.search.max_transforms = int(max_transforms)
    finder.config.search.time_limit = int(time_limit)
    finder.config.search.iteration_limit = int(iteration_limit)
    finder.config.search.return_first = bool(return_first)
    if hasattr(finder.config.search, "cutoff_cumulative"):
        finder.config.search.cutoff_cumulative = float(cutoff_cumulative)
    if hasattr(finder.config, "post_processing"):
        finder.config.post_processing.max_routes = int(max_routes_to_extract)
        finder.config.post_processing.min_routes = min(5, int(max_routes_to_extract))

excluded_list = [
    x.strip() for x in exclude_reactions_csv.split(",")
    if x.strip() and x.strip().lower() != "insert text here"
]

RETRO_SEARCH_PARAMS = {
    "ranking_mode": ranking_mode,
    "max_transforms": int(max_transforms),
    "time_limit": int(time_limit),
    "iteration_limit": int(iteration_limit),
    "return_first": bool(return_first),
    "cutoff_cumulative": float(cutoff_cumulative),
    "max_routes_to_extract": int(max_routes_to_extract),
    "target_scale_grams": float(target_scale_grams),
    "exclude_reaction_names": excluded_list,
    "use_gemini": bool(use_gemini_classifier),
}

# Typeable search settings and mode-dependent recommended presets (search runs only in Cell 5B).
import ipywidgets as widgets
_SEARCH_PRESETS = {
    "balanced": (6, 120, 100, 0.995, 10, 1.0),
    "medchem_fast": (4, 60, 75, 0.950, 5, 0.25),
    "process_scaleup": (8, 300, 350, 0.995, 15, 10.0),
    "green_safe": (6, 120, 220, 0.995, 10, 1.0),
}
_fields = ("max_transforms", "time_limit", "iteration_limit", "cutoff_cumulative", "max_routes_to_extract", "target_scale_grams")
config_widgets = {name: (widgets.BoundedFloatText if name in ("cutoff_cumulative", "target_scale_grams") else widgets.BoundedIntText)(
    value=globals()[name], min=lo, max=hi, description=name, style={"description_width": "initial"}
) for name, lo, hi in (
    ("max_transforms", 1, 12), ("time_limit", 15, 600), ("iteration_limit", 10, 500),
    ("cutoff_cumulative", 0.5, 1.0), ("max_routes_to_extract", 1, 25), ("target_scale_grams", 0.001, 100000.0),
)}
ranking_mode_widget = widgets.Dropdown(options=list(_SEARCH_PRESETS), value=ranking_mode, description="ranking_mode")

def _sync_search_settings(change=None):
    global ranking_mode
    ranking_mode = ranking_mode_widget.value
    if change and change.get("owner") is ranking_mode_widget:
        for name, value in zip(_fields, _SEARCH_PRESETS[ranking_mode]):
            config_widgets[name].value = value
    for name in _fields:
        value = config_widgets[name].value
        globals()[name] = value
        RETRO_SEARCH_PARAMS[name] = value
    RETRO_SEARCH_PARAMS["ranking_mode"] = ranking_mode
    if "finder" in globals() and finder is not None and hasattr(finder, "config"):
        for name in ("max_transforms", "time_limit", "iteration_limit", "cutoff_cumulative"):
            if hasattr(finder.config.search, name):
                setattr(finder.config.search, name, RETRO_SEARCH_PARAMS[name])
        if hasattr(finder.config, "post_processing"):
            finder.config.post_processing.max_routes = max_routes_to_extract
            finder.config.post_processing.min_routes = min(5, max_routes_to_extract)

ranking_mode_widget.observe(_sync_search_settings, names="value")
for _widget in config_widgets.values():
    _widget.observe(_sync_search_settings, names="value")
display(widgets.VBox([ranking_mode_widget, *config_widgets.values()]))


In [ ]:
# @title 5B. Launch Retrosynthesis Search { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, clear_output

run_btn = widgets.Button(
    description="🚀 Run Retrosynthesis Search",
    button_style="success",
    layout=widgets.Layout(width="280px", height="42px"),
    style={"font_weight": "bold"},
)
out_area = widgets.Output()

def _on_run_clicked(_):
    with out_area:
        clear_output(wait=True)
        params = globals().get("RETRO_SEARCH_PARAMS", {})
        if hasattr(finder, "config"):
            finder.config.search.max_transforms = int(params.get("max_transforms", max_transforms))
            finder.config.search.time_limit = int(params.get("time_limit", time_limit))
            finder.config.search.iteration_limit = int(params.get("iteration_limit", iteration_limit))
            finder.config.search.return_first = bool(params.get("return_first", return_first))
            if hasattr(finder.config.search, "cutoff_cumulative"):
                finder.config.search.cutoff_cumulative = float(params.get("cutoff_cumulative", cutoff_cumulative))
            if hasattr(finder.config, "post_processing"):
                finder.config.post_processing.max_routes = int(params.get("max_routes_to_extract", max_routes_to_extract))

        globals()["LATEST_RETRO_RESULTS"] = run_retrosynthesis_pipeline(
            finder.target_smiles,
            ranking_mode=params.get("ranking_mode", ranking_mode),
            max_transforms=int(params.get("max_transforms", max_transforms)),
            time_limit=int(params.get("time_limit", time_limit)),
            exclude_reaction_names=params.get("exclude_reaction_names", excluded_list),
            target_scale_grams=float(params.get("target_scale_grams", target_scale_grams)),
            use_gemini=bool(params.get("use_gemini", use_gemini_classifier)),
        )

run_btn.on_click(_on_run_clicked)
display(run_btn, out_area)

In [ ]:
# @title Chat with Gemini about the retrosynthesis routes
import os
import re
import html
import inspect
import urllib.parse
import requests
import markdown
import IPython
from google.colab import output, userdata
from google import genai
from google.genai import types

# 1. Build an enriched context summary of the current Target & Top 10 AiZynthFinder Routes
def _build_retrosynthesis_context():
    if "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict) and LATEST_RETRO_RESULTS.get("routes"):
        res = LATEST_RETRO_RESULTS
        lines = [
            f"Target Molecule SMILES: {res.get('target_smiles', '')}",
            f"Active Ranking Mode: {res.get('ranking_mode', 'balanced')} | Solved: {res.get('is_solved', False)}",
        ]
        if res.get("locked_substructure_smarts"):
            lines.append(f"Locked Substructure SMARTS: {res['locked_substructure_smarts']} (Atoms: {res.get('locked_atom_indices', [])})")
        if res.get("excluded_reactions"):
            lines.append(f"Excluded Reaction Filters: {', '.join(res['excluded_reactions'])}")
        lines.append("")

        for idx, r_entry in enumerate(res["routes"], start=1):
            sm = r_entry.get("starting_materials", {})
            leaf_summary = ", ".join(
                f"{p['smiles']} [{p['source']}, {p['cost_tier']}]" for p in sm.get("precursors", [])
            ) or "N/A"
            lines.append(
                f"### Route #{idx} (Composite Score: {r_entry.get('composite_score', 0.0):.4f}, "
                f"State Score: {r_entry.get('state_score', 0.0):.4f}, "
                f"Pareto Optimal: {r_entry.get('is_pareto_optimal', False)}, "
                f"Solved: {r_entry.get('is_solved', False)}, "
                f"Safety/Green Avg: {r_entry.get('avg_safety_score', 4.0):.1f}/5, "
                f"Selectivity Risks: {r_entry.get('selectivity_risk_count', 0)}, "
                f"Precursors: {sm.get('overall_cost_tier', 'N/A')} [{sm.get('in_stock_count', 0)}/{sm.get('total_leaves', 0)} in stock])"
            )
            lines.append(f"  - Starting Materials: {leaf_summary}")
            for st in r_entry.get("steps", []):
                s_idx = st.get("step_index", 1)
                name_hint = st.get("reaction_name", "Reaction Step")
                t_id = st.get("template_code", "N/A")
                fwd_smi = st.get("forward_smiles", "")
                cond = st.get("typical_conditions", "")
                safety = st.get("safety_green_score", 4)
                lines.append(
                    f"  - Step {s_idx} ({name_hint}, Template {t_id}, Safety {safety}/5): {fwd_smi} | Conditions: {cond}"
                )
                extras = []
                if st.get("selectivity_warning", "None") != "None":
                    extras.append(f"Selectivity Warning: {st['selectivity_warning']}")
                if st.get("protecting_group_note", "None") != "None":
                    extras.append(f"Protecting Group: {st['protecting_group_note']}")
                if st.get("greener_solvent_swap", "None") != "None":
                    extras.append(f"Greener Swap: {st['greener_solvent_swap']}")
                if st.get("stereochem_note", "Achiral / Racemic") != "Achiral / Racemic":
                    extras.append(f"Stereochem: {st['stereochem_note']}")
                if extras:
                    lines.append("    * " + " | ".join(extras))
        return "\n".join(lines)

    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "No target molecule or routes have been generated in AiZynthFinder yet."

    lines = [f"Target Molecule SMILES: {finder.target_smiles}\n"]
    sorted_r = sorted(
        finder.routes,
        key=lambda r: r["score"].get("state score", 0.0),
        reverse=True,
    )[:10]

    for idx, route in enumerate(sorted_r, start=1):
        score_val = route["score"].get("state score", 0.0)
        rxn_tree = route["reaction_tree"]
        lines.append(f"### Route #{idx} (State Score: {score_val:.4f}, Solved: {rxn_tree.is_solved})")
        reactions_forward = list(reversed(list(rxn_tree.reactions())))
        for s_idx, rxn_node in enumerate(reactions_forward, start=1):
            meta = getattr(rxn_node, "metadata", {}) or {}
            t_id = meta.get("template_code", meta.get("template_hash", "N/A"))
            retro_smi = rxn_node.reaction_smiles()
            prod_s, react_s = retro_smi.split(">>", 1) if ">>" in retro_smi else ("", retro_smi)
            fwd_smi = f"{react_s.strip()} >> {prod_s.strip()}"
            step_entry = gemini_lookup.get(fwd_smi) if ("gemini_lookup" in globals() and isinstance(gemini_lookup, dict)) else None
            if isinstance(step_entry, dict):
                name_hint = step_entry.get("reaction_name", "Reaction Step")
            elif hasattr(step_entry, "reaction_name"):
                name_hint = step_entry.reaction_name
            else:
                name_hint = "Step"
            lines.append(f"  - Step {s_idx} ({name_hint}, Template {t_id}): {fwd_smi}")
    return "\n".join(lines)

# 2. Agentic Python Tools for Autonomous Re-Searching, Stoichiometry/ELN & Vendor Checks (#1)
def tool_rerun_retrosynthesis(
    ranking_mode: str = "balanced",
    max_transforms: int = 6,
    exclude_reaction_keyword: str = "",
    lock_substructure_smarts: str = "",
) -> str:
    """Re-runs and re-ranks the AiZynthFinder retrosynthesis pipeline with updated search constraints.

    Args:
        ranking_mode: Multi-objective ranking profile ('balanced', 'medchem_fast', 'process_scaleup', or 'green_safe').
        max_transforms: Maximum number of retrosynthetic disconnection steps (e.g. 3 to 8).
        exclude_reaction_keyword: Comma-separated reaction or catalyst names to exclude (e.g. 'Palladium, Stille').
        lock_substructure_smarts: Optional SMARTS substructure to keep intact across all steps (e.g. 'c1ncsc1').
    """
    global LATEST_RETRO_RESULTS
    if "run_retrosynthesis_pipeline" not in globals():
        return "Error: run_retrosynthesis_pipeline is not loaded. Please run Cell 5 first."
    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "Error: No target molecule set on finder."

    exclude_list = [kw.strip() for kw in (exclude_reaction_keyword or "").split(",") if kw.strip()] or None
    lock_smarts = (lock_substructure_smarts or "").strip() or None
    kwargs = {
        "target_smiles": finder.target_smiles,
        "ranking_mode": (ranking_mode or "balanced").strip(),
        "max_transforms": int(max_transforms),
        "exclude_reaction_names": exclude_list,
        "lock_substructure_smarts": lock_smarts,
        "use_gemini": bool(globals().get("USE_GEMINI_CLASSIFIER", True)),
    }
    sig = inspect.signature(run_retrosynthesis_pipeline)
    if "render_output" in sig.parameters:
        kwargs["render_output"] = False

    updated = run_retrosynthesis_pipeline(**kwargs)
    LATEST_RETRO_RESULTS = updated
    routes = updated.get("routes", [])
    if not routes:
        return (
            f"Re-search completed (mode={ranking_mode}, max_transforms={max_transforms}, "
            f"excluded={exclude_list}, locked={lock_smarts}), but 0 routes satisfied all constraints."
        )

    summary_lines = [
        f"Updated Retrosynthesis Results ({len(routes)} routes | Mode: {updated.get('ranking_mode')} | Solved: {updated.get('is_solved')}):"
    ]
    for idx, r in enumerate(routes[:5], start=1):
        step_names = " -> ".join(s.get("reaction_name", "Step") for s in r.get("steps", []))
        sm_tier = r.get("starting_materials", {}).get("overall_cost_tier", "N/A")
        summary_lines.append(
            f"- Route #{idx}: Composite={r['composite_score']:.4f}, Pareto={r['is_pareto_optimal']}, "
            f"Steps={r['step_count']}, Safety={r['avg_safety_score']:.1f}/5, Precursors={sm_tier} | Sequence: {step_names}"
        )
    return "\n".join(summary_lines)

def tool_generate_scaled_eln_recipe(
    route_number: int,
    target_grams: float = 2.5,
    assumed_yield_per_step: float = 0.80,
) -> str:
    """Generates a complete scaled stoichiometry table (MW, eq, mmol, g, mg) and ELN lab protocol for a route.

    Args:
        route_number: 1-based route number from the active retrosynthesis results (e.g. 1 for Route #1).
        target_grams: Desired mass of final target product in grams (e.g. 2.5 or 5.0).
        assumed_yield_per_step: Fractional yield per step between 0.05 and 1.0 (default 0.80 for 80%).
    """
    if "generate_eln_markdown" not in globals():
        return "Error: generate_eln_markdown is not loaded. Please run Cell 5 first."
    return generate_eln_markdown(
        route_idx=int(route_number),
        target_scale_grams=float(target_grams),
        assumed_step_yield=float(assumed_yield_per_step),
    )

def tool_check_precursor_vendors(smiles: str) -> str:
    """Queries local stocks, PubChem PUG-REST/PUG-View, and ChEMBL for commercial vendor availability and synonyms of a SMILES.

    Args:
        smiles: Chemical SMILES string of the starting material or intermediate to look up.
    """
    smi = (smiles or "").strip()
    if not smi:
        return "Error: Empty SMILES provided."

    canon_smi = smi
    inchi_key = ""
    try:
        from rdkit import Chem
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            canon_smi = Chem.MolToSmiles(mol)
            inchi_key = Chem.MolToInchiKey(mol)
    except Exception:
        pass

    local_sources = []
    lab_keys = globals().get("CUSTOM_LAB_STOCK_INCHIKEYS", set()) or set()
    lab_smis = globals().get("CUSTOM_LAB_STOCK_SMILES", set()) or set()
    if (inchi_key and inchi_key in lab_keys) or (canon_smi in lab_smis):
        local_sources.append("Custom Lab Inventory ($0 On-Shelf)")
    if "finder" in globals() and finder is not None and hasattr(finder, "stock"):
        try:
            if hasattr(finder.stock, "smiles_in_stock") and finder.stock.smiles_in_stock(canon_smi):
                local_sources.append("Selected AiZynthFinder Stock (ZINC / Lab)")
        except Exception:
            pass

    cid = None
    synonyms = []
    vendors = []
    chembl_id = None
    enc = urllib.parse.quote(canon_smi)

    try:
        r_cid = requests.get(
            f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{enc}/cids/JSON",
            timeout=6,
        )
        if r_cid.status_code == 200:
            cids = r_cid.json().get("IdentifierList", {}).get("CID", [])
            if cids:
                cid = cids[0]
        if cid:
            r_syn = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON",
                timeout=6,
            )
            if r_syn.status_code == 200:
                info_list = r_syn.json().get("InformationList", {}).get("Information", [])
                if info_list:
                    synonyms = info_list[0].get("Synonym", [])[:6]
            r_vend = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=Chemical+Vendors",
                timeout=6,
            )
            if r_vend.status_code == 200:
                vendors = list(dict.fromkeys(re.findall(r'"SourceName"\s*:\s*"([^"]+)"', r_vend.text)))[:10]
    except Exception:
        pass

    try:
        if inchi_key:
            r_ch = requests.get(
                f"https://www.ebi.ac.uk/chembl/api/data/molecule/{inchi_key}.json",
                timeout=6,
            )
            if r_ch.status_code == 200:
                chembl_id = r_ch.json().get("molecule_chembl_id")
    except Exception:
        pass

    lines = [
        f"Precursor Vendor & Registry Report for `{canon_smi}`:",
        f"- InChIKey: `{inchi_key or 'N/A'}`",
        f"- Local Stock Status: {', '.join(local_sources) if local_sources else 'Not in local custom/ZINC stock cache'}",
        f"- PubChem CID: `{cid or 'Not found / offline'}` | ChEMBL ID: `{chembl_id or 'N/A'}`",
        f"- Common Synonyms: {', '.join(synonyms) if synonyms else 'None retrieved'}",
        f"- Commercial Vendors ({len(vendors)} sampled): {', '.join(vendors) if vendors else 'No live vendor list retrieved (check ZINC/MolPort/Sigma-Aldrich)'}",
    ]
    return "\n".join(lines)

def _route_step_forward_smiles(route_number=1, step_index=1, custom_forward_smiles=""):
    if custom_forward_smiles:
        return custom_forward_smiles.strip()
    routes = (globals().get("LATEST_RETRO_RESULTS") or {}).get("routes", [])
    if not 1 <= int(route_number) <= len(routes):
        raise ValueError("Route number is out of range; run Cell 5B first")
    steps = routes[int(route_number) - 1].get("steps", [])
    if not 1 <= int(step_index) <= len(steps):
        raise ValueError("Step index is out of range")
    return steps[int(step_index) - 1]["forward_smiles"]


def tool_render_chemical_svg(smiles_or_reaction: str, highlight_smarts: str = "", label: str = "") -> str:
    """Render a validated molecule or reaction SMILES as an inline 2D SVG chemistry card."""
    return mol_or_rxn_to_svg_html(smiles_or_reaction, highlight_smarts=highlight_smarts, label=label)


def tool_enumerate_step_byproducts(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Enumerate plausible side-products and 2D structures for a route step or custom forward reaction."""
    try:
        candidates = rdkit_enumerate_byproducts(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    return "\n".join(f"{p['origin']}: `{p['smiles']}` ({p['formula']}, {p['exact_mw']:.4f} Da) {p['svg_html']}" for p in candidates) or "No valid products identified."


def tool_predict_lcms_monitoring_ions(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Calculate monoisotopic LC-MS tracking ions for all reactants, products and side-products."""
    try:
        ions = rdkit_predict_lcms_ions(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    return "\n".join(f"{p['role']} `{p['smiles']}` {p['formula']} MW {p['exact_mw']:.4f}; " + ", ".join(f"{ion} {p[ion]:.4f}" for ion in ("[M+H]+", "[M+Na]+", "[M+NH4]+", "[M+MeCN+H]+", "[M-H]-")) + f"; {p['isotope_note']}" for p in ions) or "No valid ions identified."


def tool_compute_workup_properties(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Compute cLogP/TPSA and phase-partitioning and purification advice for a route step."""
    try:
        data = rdkit_predict_workup_properties(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    if "error" in data:
        return f"Error: {data['error']}"
    p = data["product"]
    return (f"Product `{p['smiles']}`: cLogP {p['clogp']:.2f}; TPSA {p['tpsa']:.1f} Å²; HBD/HBA {p['hbd']}/{p['hba']}; "
            f"ΔcLogP {data['delta_clogp']:.2f}, ΔTPSA {data['delta_tpsa']:.1f}. " + " ".join(data["advice"]))


def tool_modify_and_validate_step(route_number: int, step_index: int, swap_type: str, custom_reactant_smiles: str = "") -> str:
    """Validate and re-score a proposed Suzuki, N-Boc, O-TBS, acid, or custom reactant substitution."""
    import copy
    from rdkit import Chem
    if swap_type not in ("stille_to_suzuki", "install_nboc", "install_otbs", "acyl_chloride_to_acid", "custom_reactant"):
        return "Error: Unsupported swap type."
    try:
        old = _route_step_forward_smiles(route_number, step_index)
        react, product = old.split(">>")
        sources = [x.strip() for x in react.split(".") if x.strip()]
        if swap_type == "custom_reactant":
            sources = [x.strip() for x in custom_reactant_smiles.split(".") if x.strip()]
        else:
            if swap_type == "stille_to_suzuki":
                # Remove the organotin group without changing the scaffold.
                transform = "[c:1]-[Sn]>>[c:1]-B(O)O"
            elif swap_type == "install_nboc":
                transform = "[N;H1,H2;!$(NC=O):1]>>[N:1]C(=O)OC(C)(C)C"
            elif swap_type == "install_otbs":
                transform = "[O;H1;!$([O][C](=O)):1]>>[O:1][Si](C)(C)C(C)(C)C"
            else:
                transform = "[C:1](=[O:2])[Cl]>>[C:1](=[O:2])O"
            from rdkit.Chem import rdChemReactions
            rxn = rdChemReactions.ReactionFromSmarts(transform)
            transformed = False
            for i, smi in enumerate(sources):
                mol = Chem.MolFromSmiles(smi, sanitize=False)
                if mol is None or Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
                    continue
                for outcome in rxn.RunReactants((mol,), maxProducts=8):
                    candidate = outcome[0]
                    if Chem.SanitizeMol(candidate, catchErrors=True) == Chem.SanitizeFlags.SANITIZE_NONE:
                        sources[i] = Chem.MolToSmiles(candidate)
                        transformed = True
                        break
                if transformed:
                    break
            if not transformed:
                return "Error: No compatible reactant functional group for this swap."
        if not sources or any(_chem_mol(x) is None for x in sources) or any(_chem_mol(x.strip()) is None for x in product.split(".") if x.strip()):
            return "Error: Invalid or unsanitizable proposed reaction SMILES."
        new_forward = ".".join(sources) + " >> " + product.strip()
        updated = copy.deepcopy(LATEST_RETRO_RESULTS)
        step = updated["routes"][int(route_number) - 1]["steps"][int(step_index) - 1]
        inferred_name, inferred_conditions = _match_rdkit_rule(
            [_chem_mol(x) for x in sources], [_chem_mol(x.strip()) for x in product.split(".") if x.strip()]
        )
        step.update(analyze_step_heuristics(new_forward, rxn_name=inferred_name, conditions=inferred_conditions))
        if swap_type == "stille_to_suzuki":
            step["typical_conditions"] = inferred_conditions
        step["forward_smiles"] = new_forward
        step["reaction_name"] = ("Suzuki-Miyaura Cross-Coupling" if swap_type == "stille_to_suzuki" else step.get("reaction_name", "Reaction Step"))
        route = updated["routes"][int(route_number) - 1]
        route["avg_safety_score"] = sum(s.get("safety_green_score", 4) for s in route["steps"]) / len(route["steps"])
        route["selectivity_risk_count"] = sum(s.get("selectivity_warning", "None") != "None" for s in route["steps"])
        route["composite_score"] = compute_composite_route_score(route, updated.get("ranking_mode", "balanced")) if "compute_composite_route_score" in globals() else route.get("composite_score", 0)
        globals()["LATEST_RETRO_RESULTS"] = updated
        return f"Updated Route #{route_number} Step {step_index}: `{new_forward}`; safety {route['avg_safety_score']:.1f}/5, score {route['composite_score']:.4f}\n{mol_or_rxn_to_svg_html(new_forward)}"
    except (ValueError, KeyError, IndexError) as exc:
        return f"Error: {exc}"


AGENT_PYTHON_TOOLS = [
    tool_rerun_retrosynthesis,
    tool_generate_scaled_eln_recipe,
    tool_check_precursor_vendors,
    tool_render_chemical_svg,
    tool_enumerate_step_byproducts,
    tool_predict_lcms_monitoring_ions,
    tool_compute_workup_properties,
    tool_modify_and_validate_step,
]
_AGENT_TOOL_MAP = {fn.__name__: fn for fn in AGENT_PYTHON_TOOLS}

def _is_agent_tool_request(user_text: str) -> bool:
    """Determines whether the prompt asks to execute an agentic action/calculation tool rather than pure literature search."""
    t = (user_text or "").lower()
    action_patterns = (
        r"\bre-?run\b",
        r"\bre-?rank\b",
        r"\bprocess[_\s-]scale",
        r"\bmedchem[_\s-]fast\b",
        r"\bgreen[_\s-]safe\b",
        r"\bgreen\s+solvent",
        r"\bexclude\b",
        r"\bwithout\s+(?:using\s+)?[a-z0-9-]+",
        r"\bban\b",
        r"\block\b",
        r"\bfreeze\b",
        r"\bkeep\s+.*\bintact\b",
        r"\bstoichiometr",
        r"\beln\b",
        r"\brecipe\b",
        r"\b\d+(?:\.\d+)?\s*(?:g|grams|mg)\b",
        r"\bmmol\b",
        r"\b(?:byproducts?|regioisomers?|protodeboronation|lc[- ]?ms|m/z|chromatograph|workup|purification|physicochemical|svg|draw|depict|structure|swap)\b",
        r"\bvendor",
        r"\bsupplier",
        r"\bmolport\b",
        r"\bmcule\b",
        r"\bprecursor\s+(?:cost|price|availability)",
    )
    return any(re.search(pat, t) for pat in action_patterns)

def _run_offline_agent_fallback(user_text: str):
    """Executes deterministic local RDKit/pipeline tools when KDD_GEMINI_API_KEY is absent."""
    t = (user_text or "").lower()
    if any(k in t for k in ("stoichiometr", "eln", "recipe")) or re.search(r"\b\d+(?:\.\d+)?\s*(?:g|grams)\b", t):
        route_m = re.search(r"route\s*#?\s*(\d+)", t)
        grams_m = re.search(r"(\d+(?:\.\d+)?)\s*(?:g|grams)\b", t)
        yield_m = re.search(r"(\d+(?:\.\d+)?)\s*%", t)
        r_num = int(route_m.group(1)) if route_m else 1
        g_val = float(grams_m.group(1)) if grams_m else 1.0
        y_val = (float(yield_m.group(1)) / 100.0) if yield_m else 0.80
        return tool_generate_scaled_eln_recipe(route_number=r_num, target_grams=g_val, assumed_yield_per_step=y_val)
    if any(k in t for k in ("re-rank", "rerank", "re-run", "rerun", "process_scaleup", "process scale-up", "medchem_fast", "green_safe")):
        mode = "process_scaleup" if "process" in t else ("medchem_fast" if "medchem" in t else ("green_safe" if "green" in t else "balanced"))
        return tool_rerun_retrosynthesis(ranking_mode=mode)
    route_m = re.search(r"route\s*#?\s*(\d+)", t)
    step_m = re.search(r"step\s*#?\s*(\d+)", t)
    route = int(route_m.group(1)) if route_m else 1
    step = int(step_m.group(1)) if step_m else 1
    if "swap" in t or ("stille" in t and "suzuki" in t):
        return tool_modify_and_validate_step(route, step, "stille_to_suzuki")
    if "byproduct" in t or "regioisomer" in t:
        return tool_enumerate_step_byproducts(route, step) + "\n" + tool_predict_lcms_monitoring_ions(route, step)
    if "workup" in t or "purification" in t or "physicochemical" in t:
        return tool_compute_workup_properties(route, step)
    if "lc-ms" in t or "m/z" in t:
        return tool_predict_lcms_monitoring_ions(route, step)
    if "draw" in t or "svg" in t:
        smi = re.search(r"`([^`]+)`", user_text or "")
        return tool_render_chemical_svg(smi.group(1)) if smi else "Supply a backticked molecule SMILES to depict."
    return None


def _render_chemistry_markdown(text, trusted_cards=False):
    """Escape untrusted model/lookup HTML; insert only locally generated SVG cards."""
    cards = []
    if trusted_cards:
        def extract(match):
            card = match.group(0)
            if card not in globals().get("_CHEM_SVG_CARDS", set()):
                return card
            cards.append(card)
            return f"CHEMCARDPLACEHOLDER{len(cards) - 1}END"
        text = re.sub(r'<div class="chem-card">.*?</div>', extract, text, flags=re.DOTALL)
    else:
        for smi in re.findall(r"`([^`\n]{1,200})`", text):
            if ">>" in smi or re.fullmatch(r"[A-Za-z0-9@+\-\[\]()/\\#%=.:]+", smi):
                card = mol_or_rxn_to_svg_html(smi)
                if card.startswith('<div class="chem-card">'):
                    cards.append(card)
    # Do not let model-supplied Markdown introduce active links or remote images.
    text = re.sub(r"(?m)^ {0,3}\[[^]\n]+\]:\s*(?!https?://)\S+.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(r"!?\[([^]\n]*)\]\(([^)\n]*)\)",
                  lambda m: m.group(1) if m.group(0).startswith("!") or
                  urllib.parse.urlsplit(m.group(2).strip()).scheme.lower() not in ("http", "https")
                  else m.group(0), text)
    rendered = markdown.markdown(html.escape(text), extensions=["tables", "fenced_code"])
    for i, card in enumerate(cards):
        if card not in globals().get("_CHEM_SVG_CARDS", set()):
            continue
        token = f"CHEMCARDPLACEHOLDER{i}END"
        if token in rendered:
            rendered = rendered.replace(token, card)
        else:
            rendered += card
    return rendered

# 3. Persistent Multi-Turn Chat State
_CHAT_HISTORY = []  # Stores types.Content objects for multi-turn conversation

def _get_system_instruction():
    return f"""You are an autonomous AI-Agent Organic Chemist integrated into an AiZynthFinder + RDKit retrosynthesis notebook.
Your job is to help the user evaluate, re-run, re-rank, and scale up retrosynthetic routes, verify peer-reviewed/patent reaction conditions, check precursor vendors, and generate lab-ready stoichiometry protocols.

CRITICAL INSTRUCTIONS:
1. When asked to re-run/re-rank routes, exclude catalysts/reactions, lock substructures, check precursor vendors, or calculate scaled stoichiometry/ELN tables, invoke the provided Python agent tools directly.
2. When asked for literature precedents, journal references (J. Med. Chem., Org. Process Res. Dev., J. Org. Chem.), or patent procedures, use Google Search grounding and cite full references inline and at the end.
3. Provide specific stoichiometric equivalents, catalysts, greener solvent swaps, protecting group choreography, and stereochemical control notes.
4. Flag any chemoselectivity, regioselectivity, safety, or functional group compatibility risks.
5. Output chemical formulas as plain text (e.g., Pd(PPh3)4, Na2CO3). DO NOT use LaTeX or MathJax anywhere in your response.

CURRENT AIZYNTHFINDER ROUTES IN NOTEBOOK:
{_build_retrosynthesis_context()}
"""

def _invoke_with_model_fallback(client, contents, config, selected_model):
    candidate_models = [selected_model, "gemini-3.8-flash", "gemini-3-flash-preview", "gemini-2.5-flash"]
    last_err = None
    for m in dict.fromkeys(candidate_models):
        if not m:
            continue
        try:
            resp = client.models.generate_content(model=m, contents=contents, config=config)
            return resp, m, None
        except Exception as e:
            last_err = str(e)
    return None, selected_model, last_err

def _send_chat_message(user_text, selected_model, use_search):
    global _CHAT_HISTORY
    user_text = (user_text or "").strip()
    if not user_text:
        return IPython.display.JSON({"error": "Empty message"})

    api_key = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    use_agent_tools = _is_agent_tool_request(user_text)

    if not api_key:
        if use_agent_tools:
            offline_md = _run_offline_agent_fallback(user_text)
            if offline_md:
                badge = (
                    "<div style='font-size:11px; color:#137333; margin-bottom:6px;'>"
                    "🛠️ Executed deterministic local RDKit/AiZynthFinder tool (offline mode — no KDD_GEMINI_API_KEY required).</div>"
                )
                rendered_off = _render_chemistry_markdown(offline_md, trusted_cards=True)
                return IPython.display.JSON({"html": badge + rendered_off, "model": "local-rdkit-agent"})
        return IPython.display.JSON({
            "error": "Missing KDD_GEMINI_API_KEY. Please add KDD_GEMINI_API_KEY to Google Colab Secrets (🔑 icon on the left sidebar)."
        })

    client = genai.Client(api_key=api_key)
    _CHAT_HISTORY.append(types.Content(role="user", parts=[types.Part.from_text(text=user_text)]))

    tool_badges_html = ""
    response = None
    used_model = selected_model
    last_err = None

    # Route 1: Python Agent Tools (strictly separated from GoogleSearch to avoid mixed-tool API errors)
    if use_agent_tools:
        cfg_kwargs = {
            "system_instruction": _get_system_instruction(),
            "tools": AGENT_PYTHON_TOOLS,
            "temperature": 0.1,
        }
        if hasattr(types, "AutomaticFunctionCallingConfig"):
            cfg_kwargs["automatic_function_calling"] = types.AutomaticFunctionCallingConfig(disable=True)
        tool_config = types.GenerateContentConfig(**cfg_kwargs)
        response, used_model, last_err = _invoke_with_model_fallback(
            client, _CHAT_HISTORY, tool_config, selected_model
        )

        fcalls = getattr(response, "function_calls", None) if response is not None else None
        if fcalls:
            tool_outputs = []
            trusted_tool_outputs = []
            executed_names = []
            for fc in fcalls:
                fn_name = getattr(fc, "name", "")
                raw_args = dict(getattr(fc, "args", {}) or {})
                fn_obj = _AGENT_TOOL_MAP.get(fn_name)
                if fn_obj is not None:
                    try:
                        out_str = str(fn_obj(**raw_args))
                    except Exception as tool_exc:
                        out_str = f"Tool execution error in {fn_name}: {tool_exc}"
                    executed_names.append(f"{fn_name}({', '.join(f'{k}={v!r}' for k, v in raw_args.items())})")
                    tool_outputs.append(f"### Tool Output `{fn_name}`\n{out_str}")
                    if fn_name in ("tool_render_chemical_svg", "tool_enumerate_step_byproducts",
                                   "tool_modify_and_validate_step", "tool_generate_scaled_eln_recipe"):
                        trusted_tool_outputs.append(out_str)

            if executed_names:
                tool_badges_html = (
                    "<div style='font-size:11px; background:#e6f4ea; color:#137333; padding:4px 8px; "
                    "border-radius:6px; margin-bottom:8px;'>🛠️ <b>Agent Tool Executed:</b> "
                    + html.escape("; ".join(executed_names))
                    + "</div>"
                )

            combined_tool_report = "\n\n".join(tool_outputs)
            followup_contents = list(_CHAT_HISTORY) + [
                types.Content(
                    role="user",
                    parts=[types.Part.from_text(
                        text=(
                            "The requested Python chemistry tool(s) returned the following live results:\n\n"
                            + combined_tool_report
                            + "\n\nPresent and interpret these results clearly for the chemist, preserving any stoichiometry Markdown tables."
                        )
                    )],
                )
            ]
            synth_config = types.GenerateContentConfig(
                system_instruction=_get_system_instruction(),
                tools=[],
                temperature=0.2,
            )
            synth_resp, synth_model, _ = _invoke_with_model_fallback(
                client, followup_contents, synth_config, used_model
            )
            if synth_resp is not None and getattr(synth_resp, "text", None):
                response = synth_resp
                used_model = synth_model
            else:
                # Fallback directly to the deterministic tool output if follow-up generation fails
                _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=combined_tool_report)]))
                rendered_md = _render_chemistry_markdown(combined_tool_report)
                rendered_md += _render_chemistry_markdown("\n".join(trusted_tool_outputs), trusted_cards=True)
                return IPython.display.JSON({
                    "html": tool_badges_html + rendered_md,
                    "model": used_model,
                })

    # Route 2: Google Search Grounding for Live Literature & Patent Verification
    if response is None and not use_agent_tools:
        search_tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
        search_config = types.GenerateContentConfig(
            system_instruction=_get_system_instruction(),
            tools=search_tools,
            temperature=0.2,
        )
        response, used_model, last_err = _invoke_with_model_fallback(
            client, _CHAT_HISTORY, search_config, selected_model
        )

    if response is None:
        _CHAT_HISTORY.pop()  # Remove failed turn
        return IPython.display.JSON({"error": f"API Error: {last_err}"})

    reply_text = response.text or "(No text returned)"
    _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=reply_text)]))

    # Extract grounded web/literature citations if returned by Google Search tool
    sources_html = ""
    try:
        cand = response.candidates[0]
        gm = getattr(cand, "grounding_metadata", None)
        chunks = getattr(gm, "grounding_chunks", None) if gm else None
        if chunks:
            links = []
            for c in chunks:
                web = getattr(c, "web", None)
                if web and getattr(web, "uri", None):
                    title = html.escape(getattr(web, "title", None) or web.uri)
                    if urllib.parse.urlparse(web.uri).scheme in ("https", "http"):
                        links.append(f"<li><a href='{html.escape(web.uri, quote=True)}' target='_blank' rel='noopener noreferrer' style='color:#1a73e8;'>{title}</a></li>")
            if links:
                sources_html = (
                    "<div style='margin-top:10px; padding-top:8px; border-top:1px solid #e0e0e0; font-size:12px;'>"
                    "<b>📚 Grounded Literature & Web Sources:</b><ul style='margin:4px 0 0 18px; padding:0;'>"
                    + "".join(dict.fromkeys(links))
                    + "</ul></div>"
                )
    except Exception:
        pass

    note = ""
    if used_model != selected_model:
        note = f"<div style='font-size:11px; color:#b06000; margin-bottom:4px;'>ℹ️ Note: '{html.escape(selected_model)}' routed to active endpoint '{html.escape(used_model)}'.</div>"

    rendered_md = _render_chemistry_markdown(reply_text)
    if use_agent_tools and fcalls and tool_outputs:
        trusted_report = "\n".join(trusted_tool_outputs)
        if trusted_report:
            rendered_md += _render_chemistry_markdown(trusted_report, trusted_cards=True)
    return IPython.display.JSON({
        "html": note + tool_badges_html + rendered_md + sources_html,
        "model": used_model
    })

def _reset_chat_history():
    global _CHAT_HISTORY
    _CHAT_HISTORY = []
    return IPython.display.JSON({"status": "cleared"})

output.register_callback("notebook.gemini_retro_chat", _send_chat_message)
output.register_callback("notebook.gemini_retro_reset", _reset_chat_history)

# 4. Render Interactive Chat UI inside the Colab Cell Output
chat_ui_html = """
<style>
  .retro-chat-box {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #dadce0;
    border-radius: 10px;
    background: #ffffff;
    max-width: 960px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .retro-chat-header {
    background: #f8f9fa;
    padding: 12px 16px;
    border-bottom: 1px solid #dadce0;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: space-between;
    gap: 10px;
  }
  .retro-chat-messages {
    height: 440px;
    overflow-y: auto;
    padding: 16px;
    background: #fcfcfc;
    display: flex;
    flex-direction: column;
    gap: 12px;
  }
  .msg-user {
    align-self: flex-end;
    background: #e8f0fe;
    color: #174ea6;
    padding: 10px 14px;
    border-radius: 14px 14px 2px 14px;
    max-width: 78%;
    font-size: 14px;
    line-height: 1.45;
  }
  .msg-model {
    align-self: flex-start;
    background: #ffffff;
    color: #202124;
    border: 1px solid #e0e0e0;
    padding: 12px 16px;
    border-radius: 14px 14px 14px 2px;
    max-width: 88%;
    font-size: 14px;
    line-height: 1.55;
  }
  .msg-model table {
    border-collapse: collapse;
    margin: 8px 0;
    font-size: 13px;
  }
  .msg-model th, .msg-model td {
    border: 1px solid #ccc;
    padding: 6px 10px;
  }
  .retro-chat-input-area {
    padding: 12px 16px;
    border-top: 1px solid #dadce0;
    background: #f8f9fa;
    display: flex;
    gap: 10px;
    align-items: flex-end;
  }
  .retro-quick-btns {
    padding: 8px 16px;
    background: #f1f3f4;
    border-top: 1px solid #e8eaed;
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    font-size: 12px;
  }
  .quick-chip {
    background: #fff;
    border: 1px solid #dadce0;
    border-radius: 16px;
    padding: 4px 10px;
    cursor: pointer;
    color: #3c4043;
  }
  .quick-chip:hover { background: #e8f0fe; border-color: #1a73e8; color: #1a73e8; }
</style>

<div class="retro-chat-box">
  <div class="retro-chat-header">
    <div>
      <strong style="font-size:15px; color:#1a73e8;">🔬 Gemini Retrosynthesis Literature & Conditions Assistant</strong>
      <div style="font-size:12px; color:#5f6368;">Context loaded with your Target SMILES & Top 10 AiZynthFinder Routes</div>
    </div>
    <div style="display:flex; align-items:center; gap:10px;">
      <label style="font-size:12px; font-weight:600; color:#3c4043;">Model:</label>
      <select id="gemini-model-select" style="padding:5px 8px; border-radius:5px; border:1px solid #ccc; font-size:13px;">
        <option value="flash-3.8" selected>flash-3.8 (Default)</option>
        <option value="gemini-3-flash-preview">gemini-3-flash-preview</option>
        <option value="gemini-3.1-pro-preview">gemini-3.1-pro-preview</option>
        <option value="gemini-2.5-flash">gemini-2.5-flash</option>
        <option value="gemini-2.5-pro">gemini-2.5-pro</option>
      </select>
      <label style="font-size:12px; display:flex; align-items:center; gap:4px; cursor:pointer; color:#3c4043;">
        <input type="checkbox" id="gemini-search-toggle" checked /> 📚 Literature Search
      </label>
      <button id="gemini-clear-btn" style="padding:5px 10px; border:1px solid #ccc; background:#fff; border-radius:5px; cursor:pointer; font-size:12px;">
        🗑️ Clear Chat
      </button>
    </div>
  </div>

  <div id="retro-chat-messages" class="retro-chat-messages">
    <div class="msg-model">
      👋 <b>Ready!</b> I have all <b>Top 10 AiZynthFinder routes</b> for your target molecule loaded in memory.
      Ask me to search the literature for exact reaction conditions, catalysts, yields, or patent procedures for any route or step below.
    </div>
  </div>

  <div class="retro-quick-btns">
    <span style="color:#5f6368; font-weight:600; align-self:center;">Quick prompts:</span>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions, reagents, yields, and references for each step of Route #1.')">Route #1 Literature Conditions</button>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions and references for Route #6 (Suzuki coupling, SNAr, and ester hydrolysis).')">Route #6 Conditions & Refs</button>
    <button class="quick-chip" onclick="fillPrompt('Compare Route #1 (Suzuki coupling) vs Route #2 (Hantzsch thiazole cyclization) in the published literature and patents for this compound. Which has higher yield and better scalability?')">Compare Route #1 vs Route #2</button>
    <button class="quick-chip" onclick="fillPrompt('Re-run and re-rank the retrosynthesis routes in process_scaleup mode prioritizing green solvents and safety.')">⚡ Re-rank for Process Scale-Up &amp; Green Solvents</button>
    <button class="quick-chip" onclick="fillPrompt('Generate a 5.0g Stoichiometry & ELN Table for Route #1 assuming 80% step yield.')">⚖️ Generate 5.0g Stoichiometry &amp; ELN Table for Route #1</button>
    <button class="quick-chip" onclick="fillPrompt('Predict byproducts and LC-MS ions for Route #1 Step 1.')">🧪 Predict Byproducts &amp; LC-MS Ions (Route #1 Step 1)</button>
    <button class="quick-chip" onclick="fillPrompt('Compute physicochemical workup and purification plan for Route #1 Step 1.')">💧 Physicochemical Workup &amp; Purification Plan (Route #1)</button>
    <button class="quick-chip" onclick="fillPrompt('Swap Stille hazardous Step 1 of Route #1 to Suzuki and re-score.')">🔄 Swap Stille/Hazardous Step to Suzuki &amp; Re-Score</button>
  </div>

  <div class="retro-chat-input-area">
    <textarea id="retro-chat-input" rows="2" placeholder="Ask about reaction conditions, catalysts, literature precedents, or side-reactions (Press Enter to send, Shift+Enter for newline)..."
              style="flex:1; padding:10px; border:1px solid #ccc; border-radius:6px; font-family:inherit; font-size:14px; resize:vertical;"></textarea>
    <button id="retro-send-btn" style="background:#1a73e8; color:white; border:none; padding:10px 18px; border-radius:6px; font-weight:bold; cursor:pointer; height:42px;">
      Send ➤
    </button>
  </div>
</div>

<script>
function fillPrompt(text) {
  const input = document.getElementById("retro-chat-input");
  input.value = text;
  input.focus();
}

async function sendRetroMessage() {
  const inputEl = document.getElementById("retro-chat-input");
  const msgContainer = document.getElementById("retro-chat-messages");
  const modelSelect = document.getElementById("gemini-model-select").value;
  const useSearch = document.getElementById("gemini-search-toggle").checked;
  const sendBtn = document.getElementById("retro-send-btn");

  const text = inputEl.value.trim();
  if (!text) return;

  // Append user bubble
  const userDiv = document.createElement("div");
  userDiv.className = "msg-user";
  userDiv.innerText = text;
  msgContainer.appendChild(userDiv);
  inputEl.value = "";

  // Append loading bubble
  const loadingDiv = document.createElement("div");
  loadingDiv.className = "msg-model";
  loadingDiv.innerHTML = "<i>🔎 Searching chemical literature & generating response with <b>" + modelSelect + "</b>...</i>";
  msgContainer.appendChild(loadingDiv);
  msgContainer.scrollTop = msgContainer.scrollHeight;
  sendBtn.disabled = true;

  try {
    const result = await google.colab.kernel.invokeFunction(
      "notebook.gemini_retro_chat",
      [text, modelSelect, useSearch],
      {}
    );
    const data = result.data["application/json"];
    if (data.error) {
      loadingDiv.textContent = "⚠️ " + data.error;
    } else {
      loadingDiv.innerHTML = data.html;
    }
  } catch (err) {
    loadingDiv.textContent = "⚠️ Error communicating with Colab kernel: " + err;
  } finally {
    sendBtn.disabled = false;
    msgContainer.scrollTop = msgContainer.scrollHeight;
  }
}

document.getElementById("retro-send-btn").onclick = sendRetroMessage;
document.getElementById("retro-chat-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    sendRetroMessage();
  }
});

document.getElementById("gemini-clear-btn").onclick = async function() {
  await google.colab.kernel.invokeFunction("notebook.gemini_retro_reset", [], {});
  document.getElementById("retro-chat-messages").innerHTML =
    '<div class="msg-model">🧹 <b>Chat history cleared.</b> Ask a new question about any of your retrosynthetic routes!</div>';
};
</script>
"""
IPython.display.display(IPython.display.HTML(chat_ui_html))